# 9.1 ML Model Setup / Feature and Target Selection

Prepare the existing chronological train, validation, and test datasets for machine learning.

Objectives:
- Load existing train/validation/test datasets
- Verify schema consistency
- Verify date and product grain
- Separate features and target
- Exclude raw date and target from ML features
- Preserve product_id for model-specific categorical handling
- Check for missing/infinite values
- Preserve the chronological split
- Avoid target leakage

In [135]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder

import time

from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

from catboost import CatBoostRegressor

import json

In [25]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train.csv"
VALIDATION_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print()
print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)
print("Test:", TEST_PATH)

Project root: d:\All ML Projects\Retail_Demand_Forecasting
Data directory: d:\All ML Projects\Retail_Demand_Forecasting\data\processed

Train: d:\All ML Projects\Retail_Demand_Forecasting\data\processed\train.csv
Validation: d:\All ML Projects\Retail_Demand_Forecasting\data\processed\validation.csv
Test: d:\All ML Projects\Retail_Demand_Forecasting\data\processed\test.csv


In [26]:
assert TRAIN_PATH.exists(), f"Missing file: {TRAIN_PATH}"
assert VALIDATION_PATH.exists(), f"Missing file: {VALIDATION_PATH}"
assert TEST_PATH.exists(), f"Missing file: {TEST_PATH}"

print("All required split files exist.")

All required split files exist.


In [27]:
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

Train shape: (103272, 27)
Validation shape: (63109, 27)
Test shape: (81521, 27)


In [28]:
print("=" * 70)
print("TRAIN DATASET COLUMNS")
print("=" * 70)

for i, column in enumerate(train_df.columns, start=1):
    print(
        f"{i:2}. {column:<30} "
        f"dtype={str(train_df[column].dtype):<12} "
        f"missing={train_df[column].isna().sum()}"
    )

TRAIN DATASET COLUMNS
 1. date                           dtype=object       missing=0
 2. product_id                     dtype=int64        missing=0
 3. demand                         dtype=int64        missing=0
 4. year                           dtype=int64        missing=0
 5. month                          dtype=int64        missing=0
 6. day                            dtype=int64        missing=0
 7. day_of_week                    dtype=int64        missing=0
 8. week_of_year                   dtype=int64        missing=0
 9. quarter                        dtype=int64        missing=0
10. day_of_year                    dtype=int64        missing=0
11. is_weekend                     dtype=int64        missing=0
12. lag_1                          dtype=float64      missing=0
13. lag_7                          dtype=float64      missing=0
14. lag_14                         dtype=float64      missing=0
15. lag_30                         dtype=float64      missing=0
16. rolling_mean_7

In [29]:
print("\nTotal columns:", len(train_df.columns))


Total columns: 27


In [30]:
assert list(train_df.columns) == list(validation_df.columns), (
    "Train and validation schemas do not match."
)

assert list(train_df.columns) == list(test_df.columns), (
    "Train and test schemas do not match."
)

print("Schema verification passed.")
print("Train, validation, and test have identical columns and order.")

Schema verification passed.
Train, validation, and test have identical columns and order.


In [31]:
index_like_cols = [
    column
    for column in train_df.columns
    if column.lower().startswith("unnamed:")
]

print("Index-like columns:", index_like_cols)

Index-like columns: []


In [32]:
if index_like_cols:
    train_df = train_df.drop(columns=index_like_cols)
    validation_df = validation_df.drop(columns=index_like_cols)
    test_df = test_df.drop(columns=index_like_cols)

    print("Removed accidental index columns from in-memory datasets.")
else:
    print("No accidental index columns found.")

No accidental index columns found.


In [33]:
DATE_COL = "date"
TARGET_COL = "demand"
PRODUCT_COL = "product_id"

required_columns = {
    DATE_COL,
    TARGET_COL,
    PRODUCT_COL
}

for split_name, df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}.items():

    missing_columns = required_columns - set(df.columns)

    assert not missing_columns, (
        f"{split_name} is missing required columns: "
        f"{missing_columns}"
    )

print("Required columns verified.")

Required columns verified.


In [34]:
for df in [train_df, validation_df, test_df]:
    df[DATE_COL] = pd.to_datetime(
        df[DATE_COL],
        errors="raise"
    )

In [35]:
print(
    "Train:",
    train_df[DATE_COL].min().date(),
    "→",
    train_df[DATE_COL].max().date()
)

print(
    "Validation:",
    validation_df[DATE_COL].min().date(),
    "→",
    validation_df[DATE_COL].max().date()
)

print(
    "Test:",
    test_df[DATE_COL].min().date(),
    "→",
    test_df[DATE_COL].max().date()
)

Train: 2014-01-01 → 2015-05-25
Validation: 2015-05-26 → 2015-09-11
Test: 2015-09-12 → 2015-12-30


In [36]:
assert train_df[DATE_COL].max() < validation_df[DATE_COL].min(), (
    "Train and validation periods overlap."
)

assert validation_df[DATE_COL].max() < test_df[DATE_COL].min(), (
    "Validation and test periods overlap."
)

print("Chronological split verification passed.")
print()
print("Train → Validation → Test")
print("No temporal overlap detected.")

Chronological split verification passed.

Train → Validation → Test
No temporal overlap detected.


In [37]:
for split_name, df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}.items():

    duplicate_count = df.duplicated(
        subset=[DATE_COL, PRODUCT_COL]
    ).sum()

    print(
        f"{split_name:<12} duplicate date + product_id:",
        duplicate_count
    )

    assert duplicate_count == 0, (
        f"{split_name} contains duplicate date + product_id rows."
    )

train        duplicate date + product_id: 0
validation   duplicate date + product_id: 0
test         duplicate date + product_id: 0


In [38]:
NON_FEATURE_COLS = [
    TARGET_COL,
    DATE_COL
]

feature_cols = [
    column
    for column in train_df.columns
    if column not in NON_FEATURE_COLS
]

print("Excluded from ML features:")
for column in NON_FEATURE_COLS:
    print(" -", column)

print()
print("ML feature count:", len(feature_cols))

print()
print("ML features:")
for i, column in enumerate(feature_cols, start=1):
    print(f"{i:2}. {column}")

Excluded from ML features:
 - demand
 - date

ML feature count: 25

ML features:
 1. product_id
 2. year
 3. month
 4. day
 5. day_of_week
 6. week_of_year
 7. quarter
 8. day_of_year
 9. is_weekend
10. lag_1
11. lag_7
12. lag_14
13. lag_30
14. rolling_mean_7
15. rolling_mean_14
16. rolling_mean_30
17. rolling_std_7
18. expanding_mean
19. expanding_std
20. previous_count
21. product_total_demand
22. product_avg_demand
23. previous_price
24. price_change
25. product_unique_customers


In [39]:
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].copy()

X_validation = validation_df[feature_cols].copy()
y_validation = validation_df[TARGET_COL].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET_COL].copy()

In [40]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print()
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print()
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (103272, 25)
y_train: (103272,)

X_validation: (63109, 25)
y_validation: (63109,)

X_test: (81521, 25)
y_test: (81521,)


In [41]:
categorical_features = [PRODUCT_COL]

numeric_features = [
    column
    for column in feature_cols
    if column not in categorical_features
]

print("Categorical features:")
for column in categorical_features:
    print(" -", column)

print()
print("Numeric feature count:", len(numeric_features))

print()
print("Numeric features:")
for column in numeric_features:
    print(" -", column)

Categorical features:
 - product_id

Numeric feature count: 24

Numeric features:
 - year
 - month
 - day
 - day_of_week
 - week_of_year
 - quarter
 - day_of_year
 - is_weekend
 - lag_1
 - lag_7
 - lag_14
 - lag_30
 - rolling_mean_7
 - rolling_mean_14
 - rolling_mean_30
 - rolling_std_7
 - expanding_mean
 - expanding_std
 - previous_count
 - product_total_demand
 - product_avg_demand
 - previous_price
 - price_change
 - product_unique_customers


In [42]:
assert TARGET_COL not in X_train.columns
assert TARGET_COL not in X_validation.columns
assert TARGET_COL not in X_test.columns

print("Target leakage check:")
print("demand is not present in any ML feature matrix.")

Target leakage check:
demand is not present in any ML feature matrix.


In [43]:
assert DATE_COL not in X_train.columns
assert DATE_COL not in X_validation.columns
assert DATE_COL not in X_test.columns

print("Raw date exclusion check passed.")

Raw date exclusion check passed.


In [44]:
print("Missing values")
print("=" * 50)

print("X_train:", X_train.isna().sum().sum())
print("X_validation:", X_validation.isna().sum().sum())
print("X_test:", X_test.isna().sum().sum())

print()
print("Target missing values")
print("=" * 50)

print("y_train:", y_train.isna().sum())
print("y_validation:", y_validation.isna().sum())
print("y_test:", y_test.isna().sum())

Missing values
X_train: 0
X_validation: 0
X_test: 0

Target missing values
y_train: 0
y_validation: 0
y_test: 0


In [45]:
for split_name, X in {
    "train": X_train,
    "validation": X_validation,
    "test": X_test
}.items():

    numeric_data = X[numeric_features].select_dtypes(
        include=np.number
    )

    infinite_count = np.isinf(numeric_data).sum().sum()

    print(
        f"{split_name:<12} infinite numeric values:",
        infinite_count
    )

    assert infinite_count == 0, (
        f"{split_name} contains infinite numeric values."
    )

train        infinite numeric values: 0
validation   infinite numeric values: 0
test         infinite numeric values: 0


In [46]:
assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

print("X/y length verification passed.")

X/y length verification passed.


In [47]:
print("=" * 70)
print("STEP 9.1 — ML MODEL SETUP COMPLETE")
print("=" * 70)

print(f"Train rows:      {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Test rows:       {len(X_test):,}")

print()
print(f"Target: {TARGET_COL}")
print(f"Forecast grain: {DATE_COL} + {PRODUCT_COL}")

print()
print(f"Total ML features: {len(feature_cols)}")
print(f"Numeric features:  {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

print()
print("Excluded from X:")
for column in NON_FEATURE_COLS:
    print(f" - {column}")

print()
print("Categorical features:")
for column in categorical_features:
    print(f" - {column}")

print()
print("Chronological split: PRESERVED")
print("Random shuffle: NOT USED")
print("Target leakage: CHECKED")
print("Missing values: CHECKED")
print("Infinite values: CHECKED")
print("Date + product grain: CHECKED")

print()
print("STEP 9.1 COMPLETE")

STEP 9.1 — ML MODEL SETUP COMPLETE
Train rows:      103,272
Validation rows: 63,109
Test rows:       81,521

Target: demand
Forecast grain: date + product_id

Total ML features: 25
Numeric features:  24
Categorical features: 1

Excluded from X:
 - demand
 - date

Categorical features:
 - product_id

Chronological split: PRESERVED
Random shuffle: NOT USED
Target leakage: CHECKED
Missing values: CHECKED
Infinite values: CHECKED
Date + product grain: CHECKED

STEP 9.1 COMPLETE


# 9.2 Evaluation Framework

Define a consistent forecasting evaluation framework for all ML models.

Metrics:
- MAE
- RMSE
- sMAPE
- MAPE

Model selection will be performed using the validation set.

The test set will remain reserved for final model evaluation.

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [49]:
def smape(y_true, y_pred):
    """
    Symmetric Mean Absolute Percentage Error.

    Returns percentage.
    Rows where both actual and prediction are zero
    are excluded because they have no forecasting error.
    """
    
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = np.abs(y_true) + np.abs(y_pred)

    mask = denominator != 0

    if not np.any(mask):
        return 0.0

    return (
        100
        * np.mean(
            2 * np.abs(y_pred[mask] - y_true[mask])
            / denominator[mask]
        )
    )

In [50]:
def mape(y_true, y_pred):
    """
    Mean Absolute Percentage Error.

    Returns percentage.

    Observations with actual demand == 0 are excluded
    because percentage error is undefined for zero actuals.
    """
    
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if not np.any(mask):
        return 0.0

    return (
        100
        * np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        )
    )

In [51]:
def evaluate_forecast(y_true, y_pred):
    """
    Evaluate a set of forecasts using the project's
    standard forecasting metrics.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred must have the same length."
        )

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "sMAPE": smape(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
    }

In [52]:
def add_evaluation_result(
    results,
    model_name,
    y_true,
    y_pred
):
    """
    Calculate forecast metrics and append the result
    to a model comparison list.
    """

    metrics = evaluate_forecast(
        y_true,
        y_pred
    )

    results.append({
        "model": model_name,
        **metrics
    })

    return metrics

In [53]:
model_results = []

In [54]:
test_actual = np.array([10, 20, 30, 40])
test_prediction = np.array([12, 18, 33, 35])

test_metrics = evaluate_forecast(
    test_actual,
    test_prediction
)

test_metrics

{'MAE': 3.0,
 'RMSE': np.float64(3.24037034920393),
 'sMAPE': np.float64(12.891319207108682),
 'MAPE': np.float64(13.125)}

In [55]:
perfect_metrics = evaluate_forecast(
    [10, 20, 30],
    [10, 20, 30]
)

print(perfect_metrics)

{'MAE': 0.0, 'RMSE': np.float64(0.0), 'sMAPE': np.float64(0.0), 'MAPE': np.float64(0.0)}


In [56]:
baseline_validation_metrics = {
    "MAE": 36.403285,
    "RMSE": 101.897618,
    "sMAPE": 88.080540,
    "MAPE": 250.741354,
}

In [57]:
baseline_result = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    }
])

baseline_result

,model,MAE,RMSE,sMAPE,MAPE
0,Seasonal-Naive-30,36.403285,101.897618,88.08054,250.741354


In [58]:
def compare_to_baseline(model_metrics, baseline_metrics):
    """
    Calculate percentage improvement relative to the
    Seasonal-Naive-30 validation benchmark.

    Positive improvement = lower error than baseline.
    """

    comparison = {}

    for metric in baseline_metrics:
        baseline_value = baseline_metrics[metric]
        model_value = model_metrics[metric]

        if baseline_value == 0:
            comparison[f"{metric}_improvement_pct"] = np.nan
        else:
            comparison[f"{metric}_improvement_pct"] = (
                (baseline_value - model_value)
                / baseline_value
                * 100
            )

    return comparison

In [59]:
example_metrics = {
    "MAE": 30,
    "RMSE": 90,
    "sMAPE": 80,
    "MAPE": 200,
}

compare_to_baseline(
    example_metrics,
    baseline_validation_metrics
)

{'MAE_improvement_pct': 17.589854871613912,
 'RMSE_improvement_pct': 11.676051151656946,
 'sMAPE_improvement_pct': 9.174035490699762,
 'MAPE_improvement_pct': 20.236531864624137}

In [60]:
print("Evaluation policy")
print("=" * 50)

print("Model fitting:")
print(" - Train set only")

print("\nModel comparison:")
print(" - Validation set")

print("\nFinal evaluation:")
print(" - Test set")

print("\nTest set used for model selection:")
print(" - NO")

Evaluation policy
Model fitting:
 - Train set only

Model comparison:
 - Validation set

Final evaluation:
 - Test set

Test set used for model selection:
 - NO


In [61]:
print("=" * 70)
print("STEP 9.2 — EVALUATION FRAMEWORK COMPLETE")
print("=" * 70)

print("Metrics:")
print(" - MAE")
print(" - RMSE")
print(" - sMAPE")
print(" - MAPE")

print()
print("Benchmark:")
print(" - Seasonal-Naive-30")

print()
print("Validation benchmark:")
for metric, value in baseline_validation_metrics.items():
    print(f" - {metric}: {value:.6f}")

print()
print("Model selection dataset: VALIDATION")
print("Final evaluation dataset: TEST")
print("Test used for model selection: NO")

print()
print("Evaluation functions created:")
print(" - smape()")
print(" - mape()")
print(" - evaluate_forecast()")
print(" - add_evaluation_result()")
print(" - compare_to_baseline()")

print()
print("STEP 9.2 COMPLETE")

STEP 9.2 — EVALUATION FRAMEWORK COMPLETE
Metrics:
 - MAE
 - RMSE
 - sMAPE
 - MAPE

Benchmark:
 - Seasonal-Naive-30

Validation benchmark:
 - MAE: 36.403285
 - RMSE: 101.897618
 - sMAPE: 88.080540
 - MAPE: 250.741354

Model selection dataset: VALIDATION
Final evaluation dataset: TEST
Test used for model selection: NO

Evaluation functions created:
 - smape()
 - mape()
 - evaluate_forecast()
 - add_evaluation_result()
 - compare_to_baseline()

STEP 9.2 COMPLETE


# 9.3 Ridge Regression

Train a regularized linear regression model using the existing
train/validation split.

Preprocessing:
- StandardScaler for numeric features
- OneHotEncoder for product_id
- Ridge regression

The test set remains untouched.

In [62]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge

In [63]:
preprocessor_ridge = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
    ]
)

In [64]:
ridge_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_ridge
        ),
        (
            "model",
            Ridge(alpha=10.0)
        ),
    ]
)

In [65]:
print("Training Ridge Regression...")

ridge_model.fit(
    X_train,
    y_train
)

print("Ridge training complete.")

Training Ridge Regression...


Ridge training complete.


In [66]:
print("Generating validation predictions...")

ridge_validation_pred = ridge_model.predict(
    X_validation
)

print("Validation predictions generated.")
print("Prediction count:", len(ridge_validation_pred))

Generating validation predictions...
Validation predictions generated.
Prediction count: 63109


In [67]:
assert len(ridge_validation_pred) == len(y_validation)

print("Prediction length check passed.")

Prediction length check passed.


In [68]:
ridge_validation_metrics = evaluate_forecast(
    y_validation,
    ridge_validation_pred
)

print("Ridge Validation Metrics")
print("=" * 50)

for metric, value in ridge_validation_metrics.items():
    print(f"{metric}: {value:.6f}")

Ridge Validation Metrics
MAE: 20.512633
RMSE: 56.423440
sMAPE: 90.521389
MAPE: 344.070332


In [69]:
ridge_vs_baseline = compare_to_baseline(
    ridge_validation_metrics,
    baseline_validation_metrics
)

print("Ridge vs Seasonal-Naive-30")
print("=" * 50)

for metric, improvement in ridge_vs_baseline.items():
    print(f"{metric}: {improvement:.2f}%")

Ridge vs Seasonal-Naive-30
MAE_improvement_pct: 43.65%
RMSE_improvement_pct: 44.63%
sMAPE_improvement_pct: -2.77%
MAPE_improvement_pct: -37.22%


In [70]:
ridge_result = {
    "model": "Ridge",
    **ridge_validation_metrics
}

model_results.append(ridge_result)

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,MAE,RMSE,sMAPE,MAPE
0,Ridge,20.512633,56.42344,90.521389,344.070332


In [71]:
ridge_comparison_df = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    },
    ridge_result
])

ridge_comparison_df

,model,MAE,RMSE,sMAPE,MAPE
0,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354
1,Ridge,20.512633,56.423440,90.521389,344.070332


In [72]:
print("=" * 70)
print("RIDGE vs SEASONAL-NAIVE-30")
print("=" * 70)

for metric in ["MAE", "RMSE", "sMAPE", "MAPE"]:
    
    baseline_value = baseline_validation_metrics[metric]
    ridge_value = ridge_validation_metrics[metric]
    
    if ridge_value < baseline_value:
        status = "BETTER"
    else:
        status = "WORSE"
    
    print(
        f"{metric:<6} | "
        f"Baseline: {baseline_value:>12.6f} | "
        f"Ridge: {ridge_value:>12.6f} | "
        f"{status}"
    )

RIDGE vs SEASONAL-NAIVE-30
MAE    | Baseline:    36.403285 | Ridge:    20.512633 | BETTER
RMSE   | Baseline:   101.897618 | Ridge:    56.423440 | BETTER
sMAPE  | Baseline:    88.080540 | Ridge:    90.521389 | WORSE
MAPE   | Baseline:   250.741354 | Ridge:   344.070332 | WORSE


In [73]:
print("=" * 70)
print("STEP 9.3 — RIDGE REGRESSION COMPLETE")
print("=" * 70)

print("Model: Ridge Regression")
print("Preprocessing:")
print(" - Numeric → StandardScaler")
print(" - product_id → OneHotEncoder")
print(" - Unknown categories → ignored")

print()
print("Validation metrics:")

for metric, value in ridge_validation_metrics.items():
    print(f" - {metric}: {value:.6f}")

print()
print("Compared against:")
print(" - Seasonal-Naive-30")

print()
print("Test set:")
print(" - NOT USED")

print()
print("Hyperparameter tuning:")
print(" - NOT PERFORMED")
print(" - Reserved for Step 10")

print()
print("STEP 9.3 COMPLETE")

STEP 9.3 — RIDGE REGRESSION COMPLETE
Model: Ridge Regression
Preprocessing:
 - Numeric → StandardScaler
 - product_id → OneHotEncoder
 - Unknown categories → ignored

Validation metrics:
 - MAE: 20.512633
 - RMSE: 56.423440
 - sMAPE: 90.521389
 - MAPE: 344.070332

Compared against:
 - Seasonal-Naive-30

Test set:
 - NOT USED

Hyperparameter tuning:
 - NOT PERFORMED
 - Reserved for Step 10

STEP 9.3 COMPLETE


# 9.4 HistGradientBoosting Regression

Train a nonlinear gradient-boosting regression model using the
existing train/validation split.

Preprocessing:
- Numeric features → kept as numeric
- product_id → ordinal encoded
- Unknown product IDs → dedicated unknown value

The test set remains untouched.

In [74]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder

In [75]:
X_train_hgb = X_train.copy()
X_validation_hgb = X_validation.copy()
X_test_hgb = X_test.copy()

In [76]:
product_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [77]:
X_train_hgb[[PRODUCT_COL]] = product_encoder.fit_transform(
    X_train_hgb[[PRODUCT_COL]]
)

X_validation_hgb[[PRODUCT_COL]] = product_encoder.transform(
    X_validation_hgb[[PRODUCT_COL]]
)

X_test_hgb[[PRODUCT_COL]] = product_encoder.transform(
    X_test_hgb[[PRODUCT_COL]]
)

In [78]:
print("Train dtypes:")
print(X_train_hgb.dtypes)

Train dtypes:
product_id                  float64
year                          int64
month                         int64
day                           int64
day_of_week                   int64
week_of_year                  int64
quarter                       int64
day_of_year                   int64
is_weekend                    int64
lag_1                       float64
lag_7                       float64
lag_14                      float64
lag_30                      float64
rolling_mean_7              float64
rolling_mean_14             float64
rolling_mean_30             float64
rolling_std_7               float64
expanding_mean              float64
expanding_std               float64
previous_count                int64
product_total_demand        float64
product_avg_demand          float64
previous_price              float64
price_change                float64
product_unique_customers    float64
dtype: object


In [79]:
print("\nEncoded product_id sample:")
print(X_train_hgb[PRODUCT_COL].head())


Encoded product_id sample:
0    1488.0
1    2566.0
2    1928.0
3    2897.0
4     730.0
Name: product_id, dtype: float64


In [80]:
hgb_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

In [81]:
import time

print("Training HistGradientBoosting...")

start_time = time.perf_counter()

hgb_model.fit(
    X_train_hgb,
    y_train
)

elapsed_time = time.perf_counter() - start_time

print(
    f"Training complete in {elapsed_time:.2f} seconds "
    f"({elapsed_time / 60:.2f} minutes)."
)

Training HistGradientBoosting...


Training complete in 3.04 seconds (0.05 minutes).


In [82]:
print("Generating validation predictions...")

start_time = time.perf_counter()

hgb_validation_pred = hgb_model.predict(
    X_validation_hgb
)

prediction_time = time.perf_counter() - start_time

print(
    f"Validation prediction complete in "
    f"{prediction_time:.2f} seconds."
)

print("Prediction count:", len(hgb_validation_pred))

Generating validation predictions...
Validation prediction complete in 0.08 seconds.
Prediction count: 63109


In [83]:
assert len(hgb_validation_pred) == len(y_validation)

print("Prediction length check passed.")

Prediction length check passed.


In [84]:
hgb_validation_metrics = evaluate_forecast(
    y_validation,
    hgb_validation_pred
)

print("HistGradientBoosting Validation Metrics")
print("=" * 50)

for metric, value in hgb_validation_metrics.items():
    print(f"{metric}: {value:.6f}")

HistGradientBoosting Validation Metrics
MAE: 19.682154
RMSE: 56.531256
sMAPE: 84.290545
MAPE: 304.737432


In [85]:
hgb_vs_baseline = compare_to_baseline(
    hgb_validation_metrics,
    baseline_validation_metrics
)

print("HistGradientBoosting vs Seasonal-Naive-30")
print("=" * 50)

for metric, improvement in hgb_vs_baseline.items():
    print(f"{metric}: {improvement:.2f}%")

HistGradientBoosting vs Seasonal-Naive-30
MAE_improvement_pct: 45.93%
RMSE_improvement_pct: 44.52%
sMAPE_improvement_pct: 4.30%
MAPE_improvement_pct: -21.53%


In [86]:
hgb_result = {
    "model": "HistGradientBoosting",
    **hgb_validation_metrics
}

model_results.append(hgb_result)

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,MAE,RMSE,sMAPE,MAPE
0,Ridge,20.512633,56.423440,90.521389,344.070332
1,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432


In [87]:
all_validation_results = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    },
    *model_results
])

all_validation_results

,model,MAE,RMSE,sMAPE,MAPE
0,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354
1,Ridge,20.512633,56.423440,90.521389,344.070332
2,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432


In [88]:
ranking = all_validation_results.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)

print("Validation ranking")
print("=" * 70)

ranking

Validation ranking


,model,MAE,RMSE,sMAPE,MAPE
0,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
1,Ridge,20.512633,56.423440,90.521389,344.070332
2,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354


In [89]:
print("=" * 70)
print("STEP 9.4 — HISTGRADIENTBOOSTING COMPLETE")
print("=" * 70)

print("Model: HistGradientBoostingRegressor")

print()
print("Configuration:")
print(f" - learning_rate: {hgb_model.learning_rate}")
print(f" - max_iter: {hgb_model.max_iter}")
print(f" - max_leaf_nodes: {hgb_model.max_leaf_nodes}")
print(f" - min_samples_leaf: {hgb_model.min_samples_leaf}")
print(f" - l2_regularization: {hgb_model.l2_regularization}")
print(f" - random_state: {hgb_model.random_state}")

print()
print("Validation metrics:")

for metric, value in hgb_validation_metrics.items():
    print(f" - {metric}: {value:.6f}")

print()
print(f"Training time: {elapsed_time:.2f} seconds")
print(f"Validation prediction time: {prediction_time:.2f} seconds")

print()
print("Compared against:")
print(" - Seasonal-Naive-30")
print(" - Ridge")

print()
print("Test set:")
print(" - NOT USED")

print()
print("Hyperparameter tuning:")
print(" - NOT PERFORMED")
print(" - Reserved for Step 10")

print()
print("STEP 9.4 COMPLETE")

STEP 9.4 — HISTGRADIENTBOOSTING COMPLETE
Model: HistGradientBoostingRegressor

Configuration:
 - learning_rate: 0.05
 - max_iter: 300
 - max_leaf_nodes: 31
 - min_samples_leaf: 20
 - l2_regularization: 1.0
 - random_state: 42

Validation metrics:
 - MAE: 19.682154
 - RMSE: 56.531256
 - sMAPE: 84.290545
 - MAPE: 304.737432

Training time: 3.04 seconds
Validation prediction time: 0.08 seconds

Compared against:
 - Seasonal-Naive-30
 - Ridge

Test set:
 - NOT USED

Hyperparameter tuning:
 - NOT PERFORMED
 - Reserved for Step 10

STEP 9.4 COMPLETE


# 9.5 Random Forest Regression

Train a Random Forest regression model using the existing
chronological train/validation split.

Purpose:
- Evaluate a second nonlinear tree-based ML model
- Capture nonlinear relationships and feature interactions
- Compare performance against:
  - Seasonal-Naive-30 baseline
  - Ridge Regression
  - HistGradientBoosting

Preprocessing:
- Numeric features → kept as numeric
- product_id → ordinal encoded
- Unknown product IDs → dedicated unknown value

Evaluation:
- MAE
- RMSE
- sMAPE
- MAPE

Model selection:
- Validation set only
- No test-set predictions
- No hyperparameter tuning in Step 9.5
- Hyperparameter tuning is reserved for Step 10

Important:
- Preserve chronological ordering
- Do not shuffle the data
- Do not use the test set for model selection

In [90]:
from sklearn.ensemble import RandomForestRegressor

In [91]:
X_train_rf = X_train_hgb.copy()
X_validation_rf = X_validation_hgb.copy()
X_test_rf = X_test_hgb.copy()

print("Random Forest training shape:", X_train_rf.shape)
print("Random Forest validation shape:", X_validation_rf.shape)

Random Forest training shape: (103272, 25)
Random Forest validation shape: (63109, 25)


In [92]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    max_features=1.0,
    n_jobs=-1,
    random_state=42
)

In [93]:
import time

print("Training Random Forest...")
start_time = time.perf_counter()

rf_model.fit(X_train_rf, y_train)

elapsed_time = time.perf_counter() - start_time

print(
    f"Training complete in {elapsed_time:.2f} seconds "
    f"({elapsed_time / 60:.2f} minutes)."
)

Training Random Forest...


Training complete in 45.11 seconds (0.75 minutes).


In [94]:
print("Generating validation predictions...")
start_time = time.perf_counter()

rf_validation_pred = rf_model.predict(X_validation_rf)

prediction_time = time.perf_counter() - start_time

print(
    f"Validation prediction complete in "
    f"{prediction_time:.2f} seconds."
)

print("Prediction count:", len(rf_validation_pred))

Generating validation predictions...
Validation prediction complete in 0.72 seconds.
Prediction count: 63109


In [95]:
rf_validation_metrics = evaluate_forecast(
    y_validation,
    rf_validation_pred
)

print("Random Forest Validation Metrics")
print("=" * 50)

for metric, value in rf_validation_metrics.items():
    print(f"{metric}: {value:.6f}")

Random Forest Validation Metrics
MAE: 29.271655
RMSE: 67.484923
sMAPE: 95.028184
MAPE: 481.089609


In [96]:
rf_vs_baseline = compare_to_baseline(
    rf_validation_metrics,
    baseline_validation_metrics
)

print("Random Forest vs Seasonal-Naive-30")
print("=" * 50)

for metric, value in rf_vs_baseline.items():
    print(f"{metric}: {value:.2f}%")

Random Forest vs Seasonal-Naive-30
MAE_improvement_pct: 19.59%
RMSE_improvement_pct: 33.77%
sMAPE_improvement_pct: -7.89%
MAPE_improvement_pct: -91.87%


In [97]:
rf_result = {
    "model": "RandomForest",
    **rf_validation_metrics
}

model_results.append(rf_result)

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,MAE,RMSE,sMAPE,MAPE
0,Ridge,20.512633,56.423440,90.521389,344.070332
1,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
2,RandomForest,29.271655,67.484923,95.028184,481.089609


In [98]:
all_validation_results = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    },
    *model_results
])

all_validation_results

,model,MAE,RMSE,sMAPE,MAPE
0,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354
1,Ridge,20.512633,56.423440,90.521389,344.070332
2,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
3,RandomForest,29.271655,67.484923,95.028184,481.089609


In [99]:
ranking = all_validation_results.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)

ranking

,model,MAE,RMSE,sMAPE,MAPE
0,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
1,Ridge,20.512633,56.423440,90.521389,344.070332
2,RandomForest,29.271655,67.484923,95.028184,481.089609
3,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354


# 9.6 XGBoost Regression

Train an XGBoost regression model using the existing
chronological train/validation split.

Purpose:
- Evaluate a powerful gradient-boosting model for tabular forecasting
- Capture nonlinear relationships and feature interactions
- Compare performance against:
  - Seasonal-Naive-30 baseline
  - Ridge Regression
  - HistGradientBoosting
  - Random Forest

Preprocessing:
- Numeric features → kept as numeric
- product_id → use the existing ordinal encoding
- Unknown product IDs → already encoded as -1

Evaluation:
- MAE
- RMSE
- sMAPE
- MAPE

Model selection:
- Validation set only
- No test-set predictions
- No hyperparameter tuning in Step 9.6
- Hyperparameter tuning is reserved for Step 10

Important:
- Preserve chronological ordering
- Do not shuffle the data
- Do not use the test set for model selection

In [100]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [101]:
X_train_xgb = X_train_hgb.copy()
X_validation_xgb = X_validation_hgb.copy()
X_test_xgb = X_test_hgb.copy()

print("XGBoost training shape:", X_train_xgb.shape)
print("XGBoost validation shape:", X_validation_xgb.shape)

XGBoost training shape: (103272, 25)
XGBoost validation shape: (63109, 25)


In [102]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

In [103]:
import time

print("Training XGBoost...")

start_time = time.perf_counter()

xgb_model.fit(
    X_train_xgb,
    y_train
)

elapsed_time = time.perf_counter() - start_time

print(
    f"Training complete in {elapsed_time:.2f} seconds "
    f"({elapsed_time / 60:.2f} minutes)."
)

Training XGBoost...


Training complete in 2.76 seconds (0.05 minutes).


In [104]:
print("Generating validation predictions...")

start_time = time.perf_counter()

xgb_validation_pred = xgb_model.predict(
    X_validation_xgb
)

prediction_time = time.perf_counter() - start_time

print(
    f"Validation prediction complete in "
    f"{prediction_time:.2f} seconds."
)

print("Prediction count:", len(xgb_validation_pred))

Generating validation predictions...
Validation prediction complete in 0.12 seconds.
Prediction count: 63109


In [105]:
xgb_validation_metrics = evaluate_forecast(
    y_validation,
    xgb_validation_pred
)

print("XGBoost Validation Metrics")
print("=" * 50)

for metric, value in xgb_validation_metrics.items():
    print(f"{metric}: {value:.6f}")

XGBoost Validation Metrics
MAE: 24.569250
RMSE: 68.148272
sMAPE: 83.934761
MAPE: 346.208011


In [106]:
xgb_vs_baseline = compare_to_baseline(
    xgb_validation_metrics,
    baseline_validation_metrics
)

print("XGBoost vs Seasonal-Naive-30")
print("=" * 50)

for metric, value in xgb_vs_baseline.items():
    print(f"{metric}: {value:.2f}%")

XGBoost vs Seasonal-Naive-30
MAE_improvement_pct: 32.51%
RMSE_improvement_pct: 33.12%
sMAPE_improvement_pct: 4.71%
MAPE_improvement_pct: -38.07%


In [107]:
xgb_result = {
    "model": "XGBoost",
    **xgb_validation_metrics
}

model_results.append(xgb_result)

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,MAE,RMSE,sMAPE,MAPE
0,Ridge,20.512633,56.423440,90.521389,344.070332
1,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
2,RandomForest,29.271655,67.484923,95.028184,481.089609
3,XGBoost,24.569250,68.148272,83.934761,346.208011


In [108]:
all_validation_results = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    },
    *model_results
])

ranking = all_validation_results.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)

ranking

,model,MAE,RMSE,sMAPE,MAPE
0,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
1,Ridge,20.512633,56.423440,90.521389,344.070332
2,XGBoost,24.569250,68.148272,83.934761,346.208011
3,RandomForest,29.271655,67.484923,95.028184,481.089609
4,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354


# 9.7 CatBoost Regression

Train a CatBoost regression model using the existing
chronological train/validation split.

Purpose:
- Evaluate another powerful gradient-boosting model
- Handle nonlinear relationships and feature interactions
- Handle product_id as a native categorical feature
- Compare performance against:
  - Seasonal-Naive-30 baseline
  - Ridge Regression
  - HistGradientBoosting
  - Random Forest
  - XGBoost

Preprocessing:
- Numeric features → kept as numeric
- product_id → treated as a categorical feature
- No ordinal encoding of product_id for CatBoost

Evaluation:
- MAE
- RMSE
- sMAPE
- MAPE

Model selection:
- Validation set only
- No test-set predictions
- No hyperparameter tuning in Step 9.7
- Hyperparameter tuning is reserved for Step 10

Important:
- Preserve chronological ordering
- Do not shuffle the data
- Do not use the test set for model selection
- CatBoost categorical features must be specified explicitly

In [109]:
from catboost import CatBoostRegressor

print("CatBoost import successful.")

CatBoost import successful.


In [110]:
X_train_cb = X_train.copy()
X_validation_cb = X_validation.copy()
X_test_cb = X_test.copy()

print("CatBoost training shape:", X_train_cb.shape)
print("CatBoost validation shape:", X_validation_cb.shape)

CatBoost training shape: (103272, 25)
CatBoost validation shape: (63109, 25)


In [111]:
# Make product ID categorical
X_train_cb[PRODUCT_COL] = X_train_cb[PRODUCT_COL].astype(str)
X_validation_cb[PRODUCT_COL] = X_validation_cb[PRODUCT_COL].astype(str)
X_test_cb[PRODUCT_COL] = X_test_cb[PRODUCT_COL].astype(str)

In [112]:
cat_feature_indices = [
    X_train_cb.columns.get_loc(PRODUCT_COL)
]

print("Categorical feature:", PRODUCT_COL)
print("Categorical feature index:", cat_feature_indices)

Categorical feature: product_id
Categorical feature index: [0]


In [113]:
catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    l2_leaf_reg=3.0,
    random_seed=42,
    verbose=False,
    thread_count=-1
)

In [114]:
import time

print("Training CatBoost...")

start_time = time.perf_counter()

catboost_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_feature_indices
)

elapsed_time = time.perf_counter() - start_time

print(
    f"Training complete in {elapsed_time:.2f} seconds "
    f"({elapsed_time / 60:.2f} minutes)."
)

Training CatBoost...
Training complete in 36.04 seconds (0.60 minutes).


In [115]:
print("Generating validation predictions...")

start_time = time.perf_counter()

catboost_validation_pred = catboost_model.predict(
    X_validation_cb
)

prediction_time = time.perf_counter() - start_time

print(
    f"Validation prediction complete in "
    f"{prediction_time:.2f} seconds."
)

print("Prediction count:", len(catboost_validation_pred))

Generating validation predictions...
Validation prediction complete in 0.07 seconds.
Prediction count: 63109


In [116]:
catboost_validation_metrics = evaluate_forecast(
    y_validation,
    catboost_validation_pred
)

print("CatBoost Validation Metrics")
print("=" * 50)

for metric, value in catboost_validation_metrics.items():
    print(f"{metric}: {value:.6f}")

CatBoost Validation Metrics
MAE: 19.901269
RMSE: 57.684325
sMAPE: 80.717146
MAPE: 281.336673


In [117]:
catboost_vs_baseline = compare_to_baseline(
    catboost_validation_metrics,
    baseline_validation_metrics
)

print("CatBoost vs Seasonal-Naive-30")
print("=" * 50)

for metric, value in catboost_vs_baseline.items():
    print(f"{metric}: {value:.2f}%")

CatBoost vs Seasonal-Naive-30
MAE_improvement_pct: 45.33%
RMSE_improvement_pct: 43.39%
sMAPE_improvement_pct: 8.36%
MAPE_improvement_pct: -12.20%


In [118]:
catboost_result = {
    "model": "CatBoost",
    **catboost_validation_metrics
}

model_results.append(catboost_result)

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,MAE,RMSE,sMAPE,MAPE
0,Ridge,20.512633,56.423440,90.521389,344.070332
1,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
2,RandomForest,29.271655,67.484923,95.028184,481.089609
3,XGBoost,24.569250,68.148272,83.934761,346.208011
4,CatBoost,19.901269,57.684325,80.717146,281.336673


In [119]:
all_validation_results = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    },
    *model_results
])

ranking = all_validation_results.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)

ranking

,model,MAE,RMSE,sMAPE,MAPE
0,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
1,CatBoost,19.901269,57.684325,80.717146,281.336673
2,Ridge,20.512633,56.423440,90.521389,344.070332
3,XGBoost,24.569250,68.148272,83.934761,346.208011
4,RandomForest,29.271655,67.484923,95.028184,481.089609
5,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354


# 9.8 Final Model Comparison & Candidate Selection

Compare all forecasting models evaluated in Step 9 using the
validation set.

Models evaluated:
- Seasonal-Naive-30 baseline
- Ridge Regression
- HistGradientBoosting
- Random Forest
- XGBoost
- CatBoost

Primary evaluation metrics:
- MAE
- RMSE

Secondary diagnostic metrics:
- sMAPE
- MAPE

Purpose:
- Establish the final Step 9 validation leaderboard
- Identify the strongest ML candidates for hyperparameter tuning
- Exclude clearly weaker models from further tuning
- Preserve the test set for final evaluation only

Model selection policy:
- Validation data is used for model comparison
- Test data is NOT used for model selection
- No hyperparameter tuning is performed in Step 9.8

Current tuning candidates:
- HistGradientBoosting
- CatBoost

The test set remains completely untouched.

In [120]:
final_model_comparison = pd.DataFrame([
    {
        "model": "Seasonal-Naive-30",
        **baseline_validation_metrics
    },
    *model_results
])

final_model_comparison = final_model_comparison.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)

final_model_comparison

,model,MAE,RMSE,sMAPE,MAPE
0,HistGradientBoosting,19.682154,56.531256,84.290545,304.737432
1,CatBoost,19.901269,57.684325,80.717146,281.336673
2,Ridge,20.512633,56.423440,90.521389,344.070332
3,XGBoost,24.569250,68.148272,83.934761,346.208011
4,RandomForest,29.271655,67.484923,95.028184,481.089609
5,Seasonal-Naive-30,36.403285,101.897618,88.080540,250.741354


In [121]:
best_by_mae = final_model_comparison.loc[
    final_model_comparison["MAE"].idxmin()
]

best_by_rmse = final_model_comparison.loc[
    final_model_comparison["RMSE"].idxmin()
]

best_by_smape = final_model_comparison.loc[
    final_model_comparison["sMAPE"].idxmin()
]

best_by_mape = final_model_comparison.loc[
    final_model_comparison["MAPE"].idxmin()
]

print("Best model by MAE:")
print(best_by_mae["model"], "->", best_by_mae["MAE"])

print("\nBest model by RMSE:")
print(best_by_rmse["model"], "->", best_by_rmse["RMSE"])

print("\nBest model by sMAPE:")
print(best_by_smape["model"], "->", best_by_smape["sMAPE"])

print("\nBest model by MAPE:")
print(best_by_mape["model"], "->", best_by_mape["MAPE"])

Best model by MAE:
HistGradientBoosting -> 19.682153902552614

Best model by RMSE:
Ridge -> 56.42343954875243

Best model by sMAPE:
CatBoost -> 80.71714622790722

Best model by MAPE:
Seasonal-Naive-30 -> 250.741354


In [122]:
tuning_candidates = [
    "HistGradientBoosting",
    "CatBoost"
]

excluded_from_tuning = [
    "RandomForest",
    "XGBoost"
]

print("Models selected for Step 10 tuning:")
for model in tuning_candidates:
    print(" -", model)

print("\nModels not selected for primary tuning:")
for model in excluded_from_tuning:
    print(" -", model)

Models selected for Step 10 tuning:
 - HistGradientBoosting
 - CatBoost

Models not selected for primary tuning:
 - RandomForest
 - XGBoost


In [123]:
print("TEST SET USAGE VERIFICATION")
print("=" * 50)

print("Test predictions generated during Step 9: NO")
print("Test metrics calculated during Step 9: NO")
print("Test set used for model selection: NO")
print("Test set reserved for final evaluation: YES")

TEST SET USAGE VERIFICATION
Test predictions generated during Step 9: NO
Test metrics calculated during Step 9: NO
Test set used for model selection: NO
Test set reserved for final evaluation: YES


### SAVE MODELS

In [124]:
from pathlib import Path

MODEL_DIR = Path("../data/models")

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory:", MODEL_DIR.resolve())

Model directory: D:\All ML Projects\Retail_Demand_Forecasting\data\models


In [125]:
comparison_path = MODEL_DIR / "step9_model_comparison.csv"

final_model_comparison.to_csv(
    comparison_path,
    index=False
)

print("Saved:", comparison_path.resolve())

Saved: D:\All ML Projects\Retail_Demand_Forecasting\data\models\step9_model_comparison.csv


In [126]:
candidate_path = MODEL_DIR / "step9_tuning_candidates.csv"

pd.DataFrame({
    "tuning_candidate": tuning_candidates
}).to_csv(
    candidate_path,
    index=False
)

print("Saved:", candidate_path.resolve())

Saved: D:\All ML Projects\Retail_Demand_Forecasting\data\models\step9_tuning_candidates.csv


In [127]:
print("=" * 70)
print("STEP 9 — ML MODEL COMPARISON COMPLETE")
print("=" * 70)

print("\nModels evaluated:")
for model in final_model_comparison["model"]:
    print(" -", model)

print("\nBest MAE model:")
print(f" - {best_by_mae['model']} ({best_by_mae['MAE']:.6f})")

print("\nBest RMSE model:")
print(f" - {best_by_rmse['model']} ({best_by_rmse['RMSE']:.6f})")

print("\nBest sMAPE model:")
print(f" - {best_by_smape['model']} ({best_by_smape['sMAPE']:.6f})")

print("\nBest MAPE model:")
print(f" - {best_by_mape['model']} ({best_by_mape['MAPE']:.6f})")

print("\nStep 10 tuning candidates:")
for model in tuning_candidates:
    print(" -", model)

print("\nTest set:")
print(" - NOT USED")
print(" - RESERVED FOR FINAL EVALUATION")

print("\nHyperparameter tuning:")
print(" - NOT PERFORMED")
print(" - Reserved for Step 10")

print("\nSTEP 9.8 COMPLETE")

STEP 9 — ML MODEL COMPARISON COMPLETE

Models evaluated:
 - HistGradientBoosting
 - CatBoost
 - Ridge
 - XGBoost
 - RandomForest
 - Seasonal-Naive-30

Best MAE model:
 - HistGradientBoosting (19.682154)

Best RMSE model:
 - Ridge (56.423440)

Best sMAPE model:
 - CatBoost (80.717146)

Best MAPE model:
 - Seasonal-Naive-30 (250.741354)

Step 10 tuning candidates:
 - HistGradientBoosting
 - CatBoost

Test set:
 - NOT USED
 - RESERVED FOR FINAL EVALUATION

Hyperparameter tuning:
 - NOT PERFORMED
 - Reserved for Step 10

STEP 9.8 COMPLETE


### STEP 10 — HYPERPARAMETER TUNING

In [145]:
from pathlib import Path

# ============================================================
# STEP 10 — HYPERPARAMETER TUNING
# 10.1 Setup
# ============================================================

PROJECT_ROOT = Path(r"D:\All ML Projects\Retail_Demand_Forecasting")

hyperparameter_tuning_MODEL_DIR = PROJECT_ROOT / "data" / "models" / "hyperparameter_tuning"

hyperparameter_tuning_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("hyperparameter tuning model directory:")
print(hyperparameter_tuning_MODEL_DIR)

print("\nDirectory exists:", hyperparameter_tuning_MODEL_DIR.exists())

hyperparameter tuning model directory:
D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning

Directory exists: True


In [146]:
# ============================================================
# 10.1.2 — Verify Existing Train/Validation Objects
# ============================================================

required_setup_objects = [
    "X_train",
    "y_train",
    "X_validation",
    "y_validation",
    "feature_cols",
    "numeric_features",
    "categorical_features",
    "PRODUCT_COL",
]

missing_objects = [
    obj for obj in required_setup_objects
    if obj not in globals()
]

if missing_objects:
    print("Some required objects are NOT currently loaded:")
    for obj in missing_objects:
        print(" -", obj)

    print("\nThis means the Step 10 notebook kernel does not currently")
    print("have the required Step 7/9 objects in memory.")

else:
    print("All required Step 10 setup objects are available.")

    print("\nExisting train/validation shapes:")
    print("X_train:       ", X_train.shape)
    print("y_train:       ", y_train.shape)
    print("X_validation:  ", X_validation.shape)
    print("y_validation:  ", y_validation.shape)

    print("\nFeature count:", len(feature_cols))
    print("Categorical features:", categorical_features)
    print("Product column:", PRODUCT_COL)

All required Step 10 setup objects are available.

Existing train/validation shapes:
X_train:        (103272, 25)
y_train:        (103272,)
X_validation:   (63109, 25)
y_validation:   (63109,)

Feature count: 25
Categorical features: ['product_id']
Product column: product_id


In [147]:
# ============================================================
# Verify chronological split
# ============================================================

print("Expected chronological split:")
print("TRAIN:      2014-01-01 → 2015-05-25")
print("VALIDATION: 2015-05-26 → 2015-09-11")
print("TEST:       2015-09-12 → 2015-12-30")

print("\nCurrent train rows:", len(X_train))
print("Current validation rows:", len(X_validation))

if "date" in X_train.columns:
    print("\nTrain date range:")
    print(X_train["date"].min(), "→", X_train["date"].max())
else:
    print("\nRaw 'date' is not present in X_train.")
    print("This is expected because date was excluded from ML features.")

if "date" in X_validation.columns:
    print("\nValidation date range:")
    print(X_validation["date"].min(), "→", X_validation["date"].max())
else:
    print("Raw 'date' is not present in X_validation.")
    print("Existing Step 7 split will be preserved.")

Expected chronological split:
TRAIN:      2014-01-01 → 2015-05-25
VALIDATION: 2015-05-26 → 2015-09-11
TEST:       2015-09-12 → 2015-12-30

Current train rows: 103272
Current validation rows: 63109

Raw 'date' is not present in X_train.
This is expected because date was excluded from ML features.
Raw 'date' is not present in X_validation.
Existing Step 7 split will be preserved.


In [148]:
# ============================================================
# Tuning metric policy
# ============================================================

PRIMARY_METRICS = [
    "MAE",
    "RMSE",
]

SECONDARY_METRICS = [
    "sMAPE",
]

DIAGNOSTIC_METRICS = [
    "MAPE",
]

ALL_TUNING_METRICS = (
    PRIMARY_METRICS
    + SECONDARY_METRICS
    + DIAGNOSTIC_METRICS
)

print("Primary metrics:")
for metric in PRIMARY_METRICS:
    print(" -", metric)

print("\nSecondary metrics:")
for metric in SECONDARY_METRICS:
    print(" -", metric)

print("\nDiagnostic metrics:")
for metric in DIAGNOSTIC_METRICS:
    print(" -", metric)

Primary metrics:
 - MAE
 - RMSE

Secondary metrics:
 - sMAPE

Diagnostic metrics:
 - MAPE


In [149]:
# ============================================================
# Time-aware tuning policy
# ============================================================

TUNING_POLICY = {
    "split_type": "fixed_chronological_train_validation",
    "random_split": False,
    "shuffle": False,
    "test_used_for_tuning": False,
    "validation_used_for_model_selection": True,
    "primary_metrics": PRIMARY_METRICS,
    "secondary_metrics": SECONDARY_METRICS,
    "diagnostic_metrics": DIAGNOSTIC_METRICS,
}

print("Time-aware tuning policy")
print("=" * 60)

for key, value in TUNING_POLICY.items():
    print(f"{key}: {value}")

Time-aware tuning policy
split_type: fixed_chronological_train_validation
random_split: False
shuffle: False
test_used_for_tuning: False
validation_used_for_model_selection: True
primary_metrics: ['MAE', 'RMSE']
secondary_metrics: ['sMAPE']
diagnostic_metrics: ['MAPE']


In [150]:
# ============================================================
# Controlled tuning strategy
# ============================================================

TUNING_STRATEGY = {
    "strategy": "staged_controlled_search",

    "stage_1": {
        "name": "coarse_search",
        "purpose": "Explore important hyperparameter regions",
        "search_size": "small_to_moderate",
    },

    "stage_2": {
        "name": "focused_refinement",
        "purpose": "Refine around the strongest Stage 1 configurations",
        "search_size": "small",
    },

    "final_selection": {
        "primary": ["MAE", "RMSE"],
        "secondary": ["sMAPE"],
        "diagnostic": ["MAPE"],
    },

    "reproducible": True,
    "random_state": 42,
}

print("Tuning strategy")
print("=" * 60)

for stage, settings in TUNING_STRATEGY.items():
    print(f"\n{stage}:")
    if isinstance(settings, dict):
        for key, value in settings.items():
            print(f"  {key}: {value}")
    else:
        print(f"  {settings}")

Tuning strategy

strategy:
  staged_controlled_search

stage_1:
  name: coarse_search
  purpose: Explore important hyperparameter regions
  search_size: small_to_moderate

stage_2:
  name: focused_refinement
  purpose: Refine around the strongest Stage 1 configurations
  search_size: small

final_selection:
  primary: ['MAE', 'RMSE']
  secondary: ['sMAPE']
  diagnostic: ['MAPE']

reproducible:
  True

random_state:
  42


In [152]:
import json

# ============================================================
# 10.1.7 — Save Hyperparameter Tuning Setup
# ============================================================

hyperparameter_tuning_setup = {
    "project": "Retail Demand Forecasting",
    "step": "10",
    "substep": "10.1",
    "objective": "Tune best-performing ML forecasting models using time-aware validation",

    "primary_models": [
        "HistGradientBoosting",
        "CatBoost",
    ],

    "validation_strategy": {
        "type": "fixed chronological train/validation split",
        "train_rows": int(len(X_train)),
        "validation_rows": int(len(X_validation)),
        "random_split": False,
        "shuffle": False,
        "test_used": False,
    },

    "metrics": {
        "primary": PRIMARY_METRICS,
        "secondary": SECONDARY_METRICS,
        "diagnostic": DIAGNOSTIC_METRICS,
    },

    "search_strategy": "staged controlled search",

    "random_state": 42,

    "test_policy": (
        "Test set remains completely untouched during hyperparameter tuning "
        "and will only be used during final model evaluation."
    ),
}

# Save directly under data/models/
hyperparameter_tuning_path = (
    PROJECT_ROOT / "data" / "models" / "hyperparameter_tuning" / "hyperparameter_tuning.json"
)

with open(hyperparameter_tuning_path, "w", encoding="utf-8") as f:
    json.dump(hyperparameter_tuning_setup, f, indent=4)

print("Hyperparameter tuning setup saved successfully:")
print(hyperparameter_tuning_path)

print("\nFile exists:", hyperparameter_tuning_path.exists())


Hyperparameter tuning setup saved successfully:
D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\hyperparameter_tuning.json

File exists: True


In [153]:
# ============================================================
# STEP 10.2 — HistGradientBoosting Tuning
# 10.2.1 — Prepare HGB tuning data
# ============================================================

# HGB requires numerical features.
# Step 9 already created encoded HGB copies, so reuse them.

required_hgb_objects = [
    "X_train_hgb",
    "X_validation_hgb",
    "y_train",
    "y_validation",
]

missing_hgb_objects = [
    obj for obj in required_hgb_objects
    if obj not in globals()
]

if missing_hgb_objects:
    print("Missing HGB objects:")
    for obj in missing_hgb_objects:
        print(" -", obj)

    print("\nDo NOT recreate the data yet.")
    print("We need to locate the existing Step 9 HGB encoding.")
else:
    print("Existing HGB tuning data is available.")

    print("\nHGB training data:")
    print("X_train_hgb:", X_train_hgb.shape)
    print("y_train:    ", y_train.shape)

    print("\nHGB validation data:")
    print("X_validation_hgb:", X_validation_hgb.shape)
    print("y_validation:    ", y_validation.shape)

    print("\nNumber of features:", X_train_hgb.shape[1])

Existing HGB tuning data is available.

HGB training data:
X_train_hgb: (103272, 25)
y_train:     (103272,)

HGB validation data:
X_validation_hgb: (63109, 25)
y_validation:     (63109,)

Number of features: 25


In [154]:
# ============================================================
# 10.2.2 — Define Controlled HGB Search Space
# ============================================================

HGB_BASELINE_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 300,
    "max_leaf_nodes": 31,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
}

# Controlled Stage 1 configurations.
# These are intentionally limited rather than a full grid.

HGB_TUNING_CONFIGS = [

    # --------------------------------------------------------
    # Baseline configuration
    # --------------------------------------------------------
    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    # --------------------------------------------------------
    # Learning-rate / iteration trade-offs
    # --------------------------------------------------------
    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.08,
        "max_iter": 200,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.05,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    # --------------------------------------------------------
    # Tree complexity
    # --------------------------------------------------------
    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    # --------------------------------------------------------
    # Minimum leaf size / regularization
    # --------------------------------------------------------
    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 10,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 0.0,
    },

    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 5.0,
    },

    # --------------------------------------------------------
    # Combined promising alternatives
    # --------------------------------------------------------
    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.05,
        "max_iter": 500,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
    },

    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },

    {
        "learning_rate": 0.08,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20,
        "l2_regularization": 5.0,
    },
]

print("HGB Stage 1 search configurations:", len(HGB_TUNING_CONFIGS))

print("\nBaseline configuration:")
for key, value in HGB_BASELINE_PARAMS.items():
    print(f"  {key}: {value}")

print("\nSearch configuration count:")
print(len(HGB_TUNING_CONFIGS))

HGB Stage 1 search configurations: 14

Baseline configuration:
  learning_rate: 0.05
  max_iter: 300
  max_leaf_nodes: 31
  min_samples_leaf: 20
  l2_regularization: 1.0

Search configuration count:
14


In [155]:
# ============================================================
# 10.2.3 — HGB Evaluation Function
# ============================================================

import numpy as np
import time

from sklearn.metrics import mean_absolute_error, mean_squared_error


def calculate_smape(y_true, y_pred):
    """
    Calculate Symmetric Mean Absolute Percentage Error (sMAPE).

    Handles zero-demand observations safely.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = np.abs(y_true) + np.abs(y_pred)

    mask = denominator != 0

    if not np.any(mask):
        return 0.0

    smape = (
        2
        * np.abs(y_pred[mask] - y_true[mask])
        / denominator[mask]
    )

    return np.mean(smape) * 100


def calculate_mape(y_true, y_pred):
    """
    Calculate MAPE while excluding zero actual-demand observations.

    MAPE is retained only as a diagnostic metric.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if not np.any(mask):
        return np.nan

    mape = np.abs(
        (y_true[mask] - y_pred[mask])
        / y_true[mask]
    )

    return np.mean(mape) * 100


def evaluate_hgb_candidate(model, params):
    """
    Train and evaluate one HGB candidate using:

    TRAIN      -> model fitting
    VALIDATION -> model selection

    TEST       -> never used
    """

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    train_start = time.perf_counter()

    model.fit(
        X_train_hgb,
        y_train
    )

    train_time = time.perf_counter() - train_start

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------
    prediction_start = time.perf_counter()

    validation_predictions = model.predict(
        X_validation_hgb
    )

    prediction_time = time.perf_counter() - prediction_start

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------
    mae = mean_absolute_error(
        y_validation,
        validation_predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            validation_predictions
        )
    )

    smape = calculate_smape(
        y_validation,
        validation_predictions
    )

    mape = calculate_mape(
        y_validation,
        validation_predictions
    )

    return {
        **params,
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": smape,
        "MAPE": mape,
        "training_time_seconds": train_time,
        "prediction_time_seconds": prediction_time,
    }


print("HGB evaluation functions defined successfully.")

print("\nMetrics:")
print("Primary   :", PRIMARY_METRICS)
print("Secondary :", SECONDARY_METRICS)
print("Diagnostic:", DIAGNOSTIC_METRICS)

print("\nTest set used: False")

HGB evaluation functions defined successfully.

Metrics:
Primary   : ['MAE', 'RMSE']
Secondary : ['sMAPE']
Diagnostic: ['MAPE']

Test set used: False


In [156]:
# ============================================================
# 10.2.4 — Run HGB Stage 1 Controlled Search
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor
import pandas as pd

hgb_stage1_results = []

print("=" * 70)
print("HGB STAGE 1 — CONTROLLED HYPERPARAMETER SEARCH")
print("=" * 70)

print(f"\nTotal configurations: {len(HGB_TUNING_CONFIGS)}")
print("Training data:", X_train_hgb.shape)
print("Validation data:", X_validation_hgb.shape)
print("Test set used: False")

for i, params in enumerate(HGB_TUNING_CONFIGS, start=1):

    print(
        f"\n[{i}/{len(HGB_TUNING_CONFIGS)}] "
        f"lr={params['learning_rate']}, "
        f"iter={params['max_iter']}, "
        f"leaf={params['max_leaf_nodes']}, "
        f"min_leaf={params['min_samples_leaf']}, "
        f"l2={params['l2_regularization']}"
    )

    model = HistGradientBoostingRegressor(
        learning_rate=params["learning_rate"],
        max_iter=params["max_iter"],
        max_leaf_nodes=params["max_leaf_nodes"],
        min_samples_leaf=params["min_samples_leaf"],
        l2_regularization=params["l2_regularization"],
        random_state=42
    )

    result = evaluate_hgb_candidate(
        model=model,
        params=params
    )

    result["configuration_id"] = i

    hgb_stage1_results.append(result)

    print(
        f"    MAE:   {result['MAE']:.6f} | "
        f"RMSE:  {result['RMSE']:.6f} | "
        f"sMAPE: {result['sMAPE']:.6f} | "
        f"MAPE:  {result['MAPE']:.6f}"
    )

    print(
        f"    Train time: "
        f"{result['training_time_seconds']:.2f}s"
    )


# Convert results to DataFrame
hgb_stage1_results_df = pd.DataFrame(hgb_stage1_results)

# Sort primarily by MAE, then RMSE
hgb_stage1_results_df = (
    hgb_stage1_results_df
    .sort_values(
        by=["MAE", "RMSE"],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("HGB STAGE 1 COMPLETE")
print("=" * 70)

print("\nTop configurations:")
print(
    hgb_stage1_results_df[
        [
            "configuration_id",
            "learning_rate",
            "max_iter",
            "max_leaf_nodes",
            "min_samples_leaf",
            "l2_regularization",
            "MAE",
            "RMSE",
            "sMAPE",
            "MAPE",
            "training_time_seconds",
        ]
    ].head(10).to_string(index=False)
)

HGB STAGE 1 — CONTROLLED HYPERPARAMETER SEARCH

Total configurations: 14
Training data: (103272, 25)
Validation data: (63109, 25)
Test set used: False

[1/14] lr=0.05, iter=300, leaf=31, min_leaf=20, l2=1.0
    MAE:   19.682154 | RMSE:  56.531256 | sMAPE: 84.290545 | MAPE:  304.737432
    Train time: 0.78s

[2/14] lr=0.03, iter=500, leaf=31, min_leaf=20, l2=1.0
    MAE:   19.736040 | RMSE:  56.620334 | sMAPE: 84.705700 | MAPE:  307.327239
    Train time: 0.91s

[3/14] lr=0.08, iter=200, leaf=31, min_leaf=20, l2=1.0
    MAE:   19.945788 | RMSE:  57.011455 | sMAPE: 84.099715 | MAPE:  305.022258
    Train time: 0.38s

[4/14] lr=0.05, iter=500, leaf=31, min_leaf=20, l2=1.0
    MAE:   19.682154 | RMSE:  56.531256 | sMAPE: 84.290545 | MAPE:  304.737432
    Train time: 0.51s

[5/14] lr=0.05, iter=300, leaf=15, min_leaf=20, l2=1.0
    MAE:   19.234523 | RMSE:  55.996495 | sMAPE: 84.779806 | MAPE:  305.127470
    Train time: 0.40s

[6/14] lr=0.05, iter=300, leaf=63, min_leaf=20, l2=1.0
    MAE:

In [157]:
# ============================================================
# 10.2.5 — Define HGB Stage 2 Focused Search
# ============================================================

# Best Stage 1 configuration
HGB_STAGE1_BEST_PARAMS = {
    "learning_rate": 0.03,
    "max_iter": 500,
    "max_leaf_nodes": 31,
    "min_samples_leaf": 40,
    "l2_regularization": 5.0,
}

# ------------------------------------------------------------
# Focused Stage 2 configurations
# ------------------------------------------------------------
#
# We refine the promising region found in Stage 1:
#
# learning_rate:     0.02 - 0.05
# max_iter:          400 - 700
# max_leaf_nodes:    15 - 31
# min_samples_leaf:  30 - 60
# l2_regularization: 3 - 10
#
# Only a small number of combinations are tested.
# ------------------------------------------------------------

HGB_STAGE2_CONFIGS = [

    # Stage 1 best
    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },

    # Learning-rate refinement
    {
        "learning_rate": 0.02,
        "max_iter": 600,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },

    {
        "learning_rate": 0.04,
        "max_iter": 400,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },

    # Iteration refinement
    {
        "learning_rate": 0.03,
        "max_iter": 400,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },

    {
        "learning_rate": 0.03,
        "max_iter": 600,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },

    # Leaf-size refinement
    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 30,
        "l2_regularization": 5.0,
    },

    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 50,
        "l2_regularization": 5.0,
    },

    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 60,
        "l2_regularization": 5.0,
    },

    # Regularization refinement
    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 3.0,
    },

    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 7.0,
    },

    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 40,
        "l2_regularization": 10.0,
    },

    # Slightly simpler trees
    {
        "learning_rate": 0.03,
        "max_iter": 500,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 40,
        "l2_regularization": 5.0,
    },
]

print("HGB Stage 2 focused configurations:", len(HGB_STAGE2_CONFIGS))

print("\nStage 1 best configuration:")
for key, value in HGB_STAGE1_BEST_PARAMS.items():
    print(f"  {key}: {value}")

print("\nStage 2 search configurations:")
for i, params in enumerate(HGB_STAGE2_CONFIGS, start=1):
    print(
        f"{i:02d}. "
        f"lr={params['learning_rate']}, "
        f"iter={params['max_iter']}, "
        f"leaf={params['max_leaf_nodes']}, "
        f"min_leaf={params['min_samples_leaf']}, "
        f"l2={params['l2_regularization']}"
    )

HGB Stage 2 focused configurations: 12

Stage 1 best configuration:
  learning_rate: 0.03
  max_iter: 500
  max_leaf_nodes: 31
  min_samples_leaf: 40
  l2_regularization: 5.0

Stage 2 search configurations:
01. lr=0.03, iter=500, leaf=31, min_leaf=40, l2=5.0
02. lr=0.02, iter=600, leaf=31, min_leaf=40, l2=5.0
03. lr=0.04, iter=400, leaf=31, min_leaf=40, l2=5.0
04. lr=0.03, iter=400, leaf=31, min_leaf=40, l2=5.0
05. lr=0.03, iter=600, leaf=31, min_leaf=40, l2=5.0
06. lr=0.03, iter=500, leaf=31, min_leaf=30, l2=5.0
07. lr=0.03, iter=500, leaf=31, min_leaf=50, l2=5.0
08. lr=0.03, iter=500, leaf=31, min_leaf=60, l2=5.0
09. lr=0.03, iter=500, leaf=31, min_leaf=40, l2=3.0
10. lr=0.03, iter=500, leaf=31, min_leaf=40, l2=7.0
11. lr=0.03, iter=500, leaf=31, min_leaf=40, l2=10.0
12. lr=0.03, iter=500, leaf=15, min_leaf=40, l2=5.0


In [160]:
# ============================================================
# 10.2.6 — Inspect Existing HGB Evaluation Function
# ============================================================

import inspect

print(inspect.signature(evaluate_hgb_candidate))
print("\nFunction source:\n")
print(inspect.getsource(evaluate_hgb_candidate))

(model, params)

Function source:

def evaluate_hgb_candidate(model, params):
    """
    Train and evaluate one HGB candidate using:

    TRAIN      -> model fitting
    VALIDATION -> model selection

    TEST       -> never used
    """

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    train_start = time.perf_counter()

    model.fit(
        X_train_hgb,
        y_train
    )

    train_time = time.perf_counter() - train_start

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------
    prediction_start = time.perf_counter()

    validation_predictions = model.predict(
        X_validation_hgb
    )

    prediction_time = time.perf_counter() - prediction_start

    # --------------------------------------------------------
    # Validation metrics
    # ---------------------------------------

In [161]:
# ============================================================
# 10.2.6 — HGB Stage 2 Focused Search
# ============================================================

hgb_stage2_results = []

for config_id, params in enumerate(HGB_STAGE2_CONFIGS, start=1):

    print(
        f"Running HGB Stage 2 configuration "
        f"{config_id}/{len(HGB_STAGE2_CONFIGS)}..."
    )

    # Create a fresh model for this configuration
    model = HistGradientBoostingRegressor(
        **params
    )

    # evaluate_hgb_candidate returns ONE dictionary
    result = evaluate_hgb_candidate(
        model=model,
        params=params
    )

    # Add configuration metadata
    result["stage"] = "Stage 2"
    result["config_id"] = config_id

    hgb_stage2_results.append(result)


# ------------------------------------------------------------
# Convert results to DataFrame
# ------------------------------------------------------------

hgb_stage2_results_df = pd.DataFrame(
    hgb_stage2_results
)


# ------------------------------------------------------------
# Rank by primary metrics
# MAE first, RMSE second
# ------------------------------------------------------------

hgb_stage2_results_df = hgb_stage2_results_df.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HGB STAGE 2 RESULTS")
print("=" * 70)

display(hgb_stage2_results_df)


print("\n" + "=" * 70)
print("BEST HGB STAGE 2 CONFIGURATION")
print("=" * 70)

display(
    hgb_stage2_results_df.head(1)
)

Running HGB Stage 2 configuration 1/12...
Running HGB Stage 2 configuration 2/12...
Running HGB Stage 2 configuration 3/12...
Running HGB Stage 2 configuration 4/12...
Running HGB Stage 2 configuration 5/12...
Running HGB Stage 2 configuration 6/12...
Running HGB Stage 2 configuration 7/12...
Running HGB Stage 2 configuration 8/12...
Running HGB Stage 2 configuration 9/12...
Running HGB Stage 2 configuration 10/12...
Running HGB Stage 2 configuration 11/12...
Running HGB Stage 2 configuration 12/12...

HGB STAGE 2 RESULTS


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MAE,RMSE,sMAPE,MAPE,training_time_seconds,prediction_time_seconds,stage,config_id
0,0.03,500,15,40,5.0,18.737258,55.416223,83.405336,294.062569,0.662742,0.121379,Stage 2,12
1,0.03,500,31,60,5.0,19.034392,55.512241,85.301895,305.770229,0.631272,0.100980,Stage 2,8
2,0.03,500,31,40,10.0,19.060966,55.699486,83.354527,295.121027,0.885064,0.147032,Stage 2,11
3,0.03,500,31,50,5.0,19.119340,55.561696,84.604119,303.273515,0.670094,0.109486,Stage 2,7
4,0.03,500,31,30,5.0,19.159059,55.797163,84.061593,300.488357,0.780424,0.127216,Stage 2,6
5,0.03,500,31,40,3.0,19.229615,55.763008,82.918125,294.904537,0.924025,0.163192,Stage 2,9
6,0.03,500,31,40,7.0,19.240577,55.845573,83.546426,297.521691,0.863466,0.142828,Stage 2,10
7,0.03,500,31,40,5.0,19.338002,55.950510,83.218441,296.084525,1.178220,0.154388,Stage 2,1
8,0.04,400,31,40,5.0,19.385307,55.883730,84.129670,301.279516,0.679761,0.103750,Stage 2,3
9,0.02,600,31,40,5.0,19.471362,55.965443,82.719702,295.471604,1.559431,0.273396,Stage 2,2



BEST HGB STAGE 2 CONFIGURATION


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,MAE,RMSE,sMAPE,MAPE,training_time_seconds,prediction_time_seconds,stage,config_id
0,0.03,500,15,40,5.0,18.737258,55.416223,83.405336,294.062569,0.662742,0.121379,Stage 2,12


In [162]:
# ============================================================
# 10.2.7 — Save Best HGB Tuning Results
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# Best HGB configuration from Stage 2
# ------------------------------------------------------------

best_hgb_row = hgb_stage2_results_df.iloc[0]

HGB_BEST_PARAMS = {
    "learning_rate": float(best_hgb_row["learning_rate"]),
    "max_iter": int(best_hgb_row["max_iter"]),
    "max_leaf_nodes": int(best_hgb_row["max_leaf_nodes"]),
    "min_samples_leaf": int(best_hgb_row["min_samples_leaf"]),
    "l2_regularization": float(best_hgb_row["l2_regularization"]),
}

HGB_BEST_METRICS = {
    "MAE": float(best_hgb_row["MAE"]),
    "RMSE": float(best_hgb_row["RMSE"]),
    "sMAPE": float(best_hgb_row["sMAPE"]),
    "MAPE": float(best_hgb_row["MAPE"]),
}


# ------------------------------------------------------------
# Save directory
# ------------------------------------------------------------

TUNING_DIR = Path(
    r"D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning"
)

TUNING_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Save Stage 2 results
# ------------------------------------------------------------

hgb_stage2_results_path = TUNING_DIR / "hgb_stage2_results.csv"

hgb_stage2_results_df.to_csv(
    hgb_stage2_results_path,
    index=False
)


# ------------------------------------------------------------
# Save best HGB configuration + metrics
# ------------------------------------------------------------

hgb_best_path = TUNING_DIR / "hgb_best_params.json"

hgb_best_record = {
    "model": "HistGradientBoostingRegressor",
    "selection_metric": "MAE",
    "secondary_selection_metric": "RMSE",
    "best_params": HGB_BEST_PARAMS,
    "validation_metrics": HGB_BEST_METRICS,
    "test_used_for_tuning": False,
    "validation_used_for_selection": True,
    "random_split": False,
    "shuffle": False,
    "random_state": 42
}

with open(hgb_best_path, "w") as f:
    json.dump(
        hgb_best_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# Display confirmation
# ------------------------------------------------------------

print("=" * 70)
print("BEST HGB CONFIGURATION SAVED")
print("=" * 70)

print("\nBest parameters:")
for key, value in HGB_BEST_PARAMS.items():
    print(f"{key}: {value}")

print("\nValidation metrics:")
for key, value in HGB_BEST_METRICS.items():
    print(f"{key}: {value:.6f}")

print("\nSaved files:")
print(f"Stage 2 results: {hgb_stage2_results_path}")
print(f"Best parameters: {hgb_best_path}")

print("\nTest set used for tuning: False")

BEST HGB CONFIGURATION SAVED

Best parameters:
learning_rate: 0.03
max_iter: 500
max_leaf_nodes: 15
min_samples_leaf: 40
l2_regularization: 5.0

Validation metrics:
MAE: 18.737258
RMSE: 55.416223
sMAPE: 83.405336
MAPE: 294.062569

Saved files:
Stage 2 results: D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\hgb_stage2_results.csv
Best parameters: D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\hgb_best_params.json

Test set used for tuning: False


In [163]:
# ============================================================
# 10.3.1 — Verify CatBoost Tuning Data
# ============================================================

print("=" * 70)
print("CATBOOST TUNING DATA VERIFICATION")
print("=" * 70)

required_objects = [
    "X_train",
    "y_train",
    "X_validation",
    "y_validation",
    "categorical_features",
    "PRODUCT_COL"
]

print("\nRequired objects:")
for obj_name in required_objects:
    print(f"{obj_name}: {'EXISTS' if obj_name in globals() else 'MISSING'}")


print("\nShapes:")

if "X_train" in globals():
    print("X_train:", X_train.shape)

if "y_train" in globals():
    print("y_train:", y_train.shape)

if "X_validation" in globals():
    print("X_validation:", X_validation.shape)

if "y_validation" in globals():
    print("y_validation:", y_validation.shape)


print("\nCategorical features:")
if "categorical_features" in globals():
    print(categorical_features)

print("\nProduct column:")
if "PRODUCT_COL" in globals():
    print(PRODUCT_COL)


# ------------------------------------------------------------
# Basic consistency checks
# ------------------------------------------------------------

if all(obj in globals() for obj in required_objects):

    assert len(X_train) == len(y_train)
    assert len(X_validation) == len(y_validation)

    assert PRODUCT_COL in X_train.columns
    assert PRODUCT_COL in X_validation.columns

    print("\n" + "=" * 70)
    print("VERIFICATION PASSED")
    print("=" * 70)

    print("\nCatBoost can use:")
    print("- X_train + y_train")
    print("- X_validation + y_validation")
    print(f"- Categorical feature: {PRODUCT_COL}")

    print("\nTest set will NOT be used for tuning.")

else:
    print("\n" + "=" * 70)
    print("VERIFICATION FAILED — REQUIRED OBJECTS ARE MISSING")
    print("=" * 70)

CATBOOST TUNING DATA VERIFICATION

Required objects:
X_train: EXISTS
y_train: EXISTS
X_validation: EXISTS
y_validation: EXISTS
categorical_features: EXISTS
PRODUCT_COL: EXISTS

Shapes:
X_train: (103272, 25)
y_train: (103272,)
X_validation: (63109, 25)
y_validation: (63109,)

Categorical features:
['product_id']

Product column:
product_id

VERIFICATION PASSED

CatBoost can use:
- X_train + y_train
- X_validation + y_validation
- Categorical feature: product_id

Test set will NOT be used for tuning.


In [164]:
# ============================================================
# 10.3.2 — CatBoost Stage 1 Controlled Search
# ============================================================

CATBOOST_STAGE1_CONFIGS = [
    # 01 — Original baseline
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0
    },

    # 02 — Lower learning rate + more iterations
    {
        "iterations": 800,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 3.0
    },

    # 03 — Higher learning rate + fewer iterations
    {
        "iterations": 300,
        "learning_rate": 0.08,
        "depth": 6,
        "l2_leaf_reg": 3.0
    },

    # 04 — More iterations
    {
        "iterations": 800,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0
    },

    # 05 — Shallower trees
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 4,
        "l2_leaf_reg": 3.0
    },

    # 06 — Deeper trees
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 8,
        "l2_leaf_reg": 3.0
    },

    # 07 — Stronger L2 regularization
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0
    },

    # 08 — Weaker L2 regularization
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 1.0
    },

    # 09 — Lower learning rate + deeper trees
    {
        "iterations": 800,
        "learning_rate": 0.03,
        "depth": 8,
        "l2_leaf_reg": 3.0
    },

    # 10 — Lower learning rate + stronger regularization
    {
        "iterations": 800,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 5.0
    },

    # 11 — Moderate learning rate + shallow trees
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 4,
        "l2_leaf_reg": 3.0
    },

    # 12 — Moderate learning rate + deeper trees
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 3.0
    },

    # 13 — Deeper trees + stronger regularization
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 5.0
    },

    # 14 — Shallower trees + stronger regularization
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 4,
        "l2_leaf_reg": 5.0
    }
]


print("=" * 70)
print("CATBOOST STAGE 1 SEARCH CONFIGURATIONS")
print("=" * 70)

print(f"\nTotal configurations: {len(CATBOOST_STAGE1_CONFIGS)}")

for i, params in enumerate(CATBOOST_STAGE1_CONFIGS, start=1):
    print(
        f"{i:02d}. "
        f"iterations={params['iterations']}, "
        f"lr={params['learning_rate']}, "
        f"depth={params['depth']}, "
        f"l2={params['l2_leaf_reg']}"
    )

CATBOOST STAGE 1 SEARCH CONFIGURATIONS

Total configurations: 14
01. iterations=500, lr=0.05, depth=6, l2=3.0
02. iterations=800, lr=0.03, depth=6, l2=3.0
03. iterations=300, lr=0.08, depth=6, l2=3.0
04. iterations=800, lr=0.05, depth=6, l2=3.0
05. iterations=500, lr=0.05, depth=4, l2=3.0
06. iterations=500, lr=0.05, depth=8, l2=3.0
07. iterations=500, lr=0.05, depth=6, l2=5.0
08. iterations=500, lr=0.05, depth=6, l2=1.0
09. iterations=800, lr=0.03, depth=8, l2=3.0
10. iterations=800, lr=0.03, depth=6, l2=5.0
11. iterations=600, lr=0.04, depth=4, l2=3.0
12. iterations=600, lr=0.04, depth=8, l2=3.0
13. iterations=600, lr=0.04, depth=8, l2=5.0
14. iterations=600, lr=0.04, depth=4, l2=5.0


In [165]:
# ============================================================
# 10.3.3 — CatBoost Candidate Evaluation Function
# ============================================================

import time
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor


def evaluate_catboost_candidate(model, params):
    """
    Train and evaluate one CatBoost candidate using:

    TRAIN      -> model fitting
    VALIDATION -> model selection

    TEST       -> never used
    """

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    train_start = time.perf_counter()

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_features
    )

    train_time = time.perf_counter() - train_start

    # --------------------------------------------------------
    # Validation prediction
    # --------------------------------------------------------
    prediction_start = time.perf_counter()

    validation_predictions = model.predict(
        X_validation
    )

    prediction_time = time.perf_counter() - prediction_start

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------
    mae = mean_absolute_error(
        y_validation,
        validation_predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            validation_predictions
        )
    )

    smape = calculate_smape(
        y_validation,
        validation_predictions
    )

    mape = calculate_mape(
        y_validation,
        validation_predictions
    )

    return {
        **params,
        "MAE": mae,
        "RMSE": rmse,
        "sMAPE": smape,
        "MAPE": mape,
        "training_time_seconds": train_time,
        "prediction_time_seconds": prediction_time,
    }


print("=" * 70)
print("CATBOOST EVALUATION FUNCTION READY")
print("=" * 70)

print("\nTraining data:", X_train.shape)
print("Validation data:", X_validation.shape)
print("Categorical features:", categorical_features)
print("\nTest set: NOT USED")

CATBOOST EVALUATION FUNCTION READY

Training data: (103272, 25)
Validation data: (63109, 25)
Categorical features: ['product_id']

Test set: NOT USED


In [166]:
# ============================================================
# 10.3.4 — CatBoost Stage 1 Search
# ============================================================

catboost_stage1_results = []

for config_id, params in enumerate(CATBOOST_STAGE1_CONFIGS, start=1):

    print(
        f"Running CatBoost Stage 1 configuration "
        f"{config_id}/{len(CATBOOST_STAGE1_CONFIGS)}..."
    )

    # Create a fresh CatBoost model
    model = CatBoostRegressor(
        **params,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    # Evaluate using train + validation only
    result = evaluate_catboost_candidate(
        model=model,
        params=params
    )

    # Add search metadata
    result["stage"] = "Stage 1"
    result["config_id"] = config_id

    catboost_stage1_results.append(result)


# ------------------------------------------------------------
# Convert results to DataFrame
# ------------------------------------------------------------

catboost_stage1_results_df = pd.DataFrame(
    catboost_stage1_results
)


# ------------------------------------------------------------
# Rank by primary metrics
# MAE first, RMSE second
# ------------------------------------------------------------

catboost_stage1_results_df = catboost_stage1_results_df.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)


# ------------------------------------------------------------
# Display all results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATBOOST STAGE 1 RESULTS")
print("=" * 70)

display(catboost_stage1_results_df)


# ------------------------------------------------------------
# Display best configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST CATBOOST STAGE 1 CONFIGURATION")
print("=" * 70)

display(
    catboost_stage1_results_df.head(1)
)

Running CatBoost Stage 1 configuration 1/14...
Running CatBoost Stage 1 configuration 2/14...
Running CatBoost Stage 1 configuration 3/14...
Running CatBoost Stage 1 configuration 4/14...
Running CatBoost Stage 1 configuration 5/14...
Running CatBoost Stage 1 configuration 6/14...
Running CatBoost Stage 1 configuration 7/14...
Running CatBoost Stage 1 configuration 8/14...
Running CatBoost Stage 1 configuration 9/14...
Running CatBoost Stage 1 configuration 10/14...
Running CatBoost Stage 1 configuration 11/14...
Running CatBoost Stage 1 configuration 12/14...
Running CatBoost Stage 1 configuration 13/14...
Running CatBoost Stage 1 configuration 14/14...

CATBOOST STAGE 1 RESULTS


,iterations,learning_rate,depth,l2_leaf_reg,MAE,RMSE,sMAPE,MAPE,training_time_seconds,prediction_time_seconds,stage,config_id
0,600,0.04,8,5.0,19.315817,56.684225,79.894261,272.502008,57.916358,0.073824,Stage 1,13
1,600,0.04,4,3.0,19.362897,56.983589,80.781942,277.831302,28.059802,0.066167,Stage 1,11
2,600,0.04,4,5.0,19.617538,57.374463,81.043956,280.320590,28.348588,0.045724,Stage 1,14
3,500,0.05,4,3.0,19.837253,57.576433,81.256164,283.357975,24.248086,0.045657,Stage 1,5
4,500,0.05,6,5.0,19.896133,57.807669,80.631038,280.822673,36.026282,0.080104,Stage 1,7
5,500,0.05,6,3.0,19.901269,57.684325,80.717146,281.336673,37.260397,0.081656,Stage 1,1
6,800,0.03,6,3.0,19.913592,57.502004,80.739560,283.131322,56.920386,0.072730,Stage 1,2
7,800,0.03,8,3.0,19.983538,58.253844,80.279619,281.013267,78.097651,0.086386,Stage 1,9
8,300,0.08,6,3.0,20.063091,57.658550,81.091117,284.232932,23.059606,0.056981,Stage 1,3
9,500,0.05,8,3.0,20.082426,58.339916,80.675614,286.756607,49.363528,0.072790,Stage 1,6



BEST CATBOOST STAGE 1 CONFIGURATION


,iterations,learning_rate,depth,l2_leaf_reg,MAE,RMSE,sMAPE,MAPE,training_time_seconds,prediction_time_seconds,stage,config_id
0,600,0.04,8,5.0,19.315817,56.684225,79.894261,272.502008,57.916358,0.073824,Stage 1,13


In [167]:
# ============================================================
# 10.3.5 — CatBoost Stage 2 Focused Search
# ============================================================

CATBOOST_STAGE1_BEST_PARAMS = {
    "iterations": 600,
    "learning_rate": 0.04,
    "depth": 8,
    "l2_leaf_reg": 5.0
}


CATBOOST_STAGE2_CONFIGS = [

    # 01 — Stage 1 best
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 5.0
    },

    # 02 — Slightly lower learning rate + more iterations
    {
        "iterations": 700,
        "learning_rate": 0.035,
        "depth": 8,
        "l2_leaf_reg": 5.0
    },

    # 03 — Slightly higher learning rate + fewer iterations
    {
        "iterations": 500,
        "learning_rate": 0.05,
        "depth": 8,
        "l2_leaf_reg": 5.0
    },

    # 04 — More iterations
    {
        "iterations": 800,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 5.0
    },

    # 05 — Fewer iterations
    {
        "iterations": 500,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 5.0
    },

    # 06 — Slightly shallower trees
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 7,
        "l2_leaf_reg": 5.0
    },

    # 07 — Slightly deeper trees
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 9,
        "l2_leaf_reg": 5.0
    },

    # 08 — Lower L2 regularization
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 3.0
    },

    # 09 — Higher L2 regularization
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 7.0
    },

    # 10 — Stronger L2 regularization
    {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 8,
        "l2_leaf_reg": 10.0
    },

    # 11 — Lower learning rate + more iterations + depth 9
    {
        "iterations": 700,
        "learning_rate": 0.035,
        "depth": 9,
        "l2_leaf_reg": 5.0
    },

    # 12 — Moderate learning rate + depth 7 + stronger L2
    {
        "iterations": 700,
        "learning_rate": 0.035,
        "depth": 7,
        "l2_leaf_reg": 7.0
    }
]


print("=" * 70)
print("CATBOOST STAGE 2 FOCUSED SEARCH CONFIGURATIONS")
print("=" * 70)

print(
    f"\nStage 1 best configuration:\n"
    f"iterations: {CATBOOST_STAGE1_BEST_PARAMS['iterations']}\n"
    f"learning_rate: {CATBOOST_STAGE1_BEST_PARAMS['learning_rate']}\n"
    f"depth: {CATBOOST_STAGE1_BEST_PARAMS['depth']}\n"
    f"l2_leaf_reg: {CATBOOST_STAGE1_BEST_PARAMS['l2_leaf_reg']}"
)

print(
    f"\nTotal Stage 2 configurations: "
    f"{len(CATBOOST_STAGE2_CONFIGS)}"
)

print("\nStage 2 search configurations:")

for i, params in enumerate(CATBOOST_STAGE2_CONFIGS, start=1):
    print(
        f"{i:02d}. "
        f"iterations={params['iterations']}, "
        f"lr={params['learning_rate']}, "
        f"depth={params['depth']}, "
        f"l2={params['l2_leaf_reg']}"
    )

CATBOOST STAGE 2 FOCUSED SEARCH CONFIGURATIONS

Stage 1 best configuration:
iterations: 600
learning_rate: 0.04
depth: 8
l2_leaf_reg: 5.0

Total Stage 2 configurations: 12

Stage 2 search configurations:
01. iterations=600, lr=0.04, depth=8, l2=5.0
02. iterations=700, lr=0.035, depth=8, l2=5.0
03. iterations=500, lr=0.05, depth=8, l2=5.0
04. iterations=800, lr=0.04, depth=8, l2=5.0
05. iterations=500, lr=0.04, depth=8, l2=5.0
06. iterations=600, lr=0.04, depth=7, l2=5.0
07. iterations=600, lr=0.04, depth=9, l2=5.0
08. iterations=600, lr=0.04, depth=8, l2=3.0
09. iterations=600, lr=0.04, depth=8, l2=7.0
10. iterations=600, lr=0.04, depth=8, l2=10.0
11. iterations=700, lr=0.035, depth=9, l2=5.0
12. iterations=700, lr=0.035, depth=7, l2=7.0


In [168]:
# ============================================================
# 10.3.6 — CatBoost Stage 2 Focused Search
# ============================================================

catboost_stage2_results = []

for config_id, params in enumerate(CATBOOST_STAGE2_CONFIGS, start=1):

    print(
        f"Running CatBoost Stage 2 configuration "
        f"{config_id}/{len(CATBOOST_STAGE2_CONFIGS)}..."
    )

    # Create a fresh CatBoost model for each configuration
    model = CatBoostRegressor(
        **params,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1
    )

    # evaluate_catboost_candidate returns ONE dictionary
    result = evaluate_catboost_candidate(
        model=model,
        params=params
    )

    # Add search metadata
    result["stage"] = "Stage 2"
    result["config_id"] = config_id

    catboost_stage2_results.append(result)


# ------------------------------------------------------------
# Convert results to DataFrame
# ------------------------------------------------------------

catboost_stage2_results_df = pd.DataFrame(
    catboost_stage2_results
)


# ------------------------------------------------------------
# Rank by primary metrics
# MAE first, RMSE second
# ------------------------------------------------------------

catboost_stage2_results_df = catboost_stage2_results_df.sort_values(
    by=["MAE", "RMSE"],
    ascending=[True, True]
).reset_index(drop=True)


# ------------------------------------------------------------
# Display all results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATBOOST STAGE 2 RESULTS")
print("=" * 70)

display(catboost_stage2_results_df)


# ------------------------------------------------------------
# Display best configuration
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEST CATBOOST STAGE 2 CONFIGURATION")
print("=" * 70)

display(
    catboost_stage2_results_df.head(1)
)

Running CatBoost Stage 2 configuration 1/12...
Running CatBoost Stage 2 configuration 2/12...
Running CatBoost Stage 2 configuration 3/12...
Running CatBoost Stage 2 configuration 4/12...
Running CatBoost Stage 2 configuration 5/12...
Running CatBoost Stage 2 configuration 6/12...
Running CatBoost Stage 2 configuration 7/12...
Running CatBoost Stage 2 configuration 8/12...
Running CatBoost Stage 2 configuration 9/12...
Running CatBoost Stage 2 configuration 10/12...
Running CatBoost Stage 2 configuration 11/12...
Running CatBoost Stage 2 configuration 12/12...

CATBOOST STAGE 2 RESULTS


,iterations,learning_rate,depth,l2_leaf_reg,MAE,RMSE,sMAPE,MAPE,training_time_seconds,prediction_time_seconds,stage,config_id
0,600,0.040,9,5.0,19.239601,56.555438,80.110475,274.477167,72.069118,0.139299,Stage 2,7
1,600,0.040,8,7.0,19.255597,56.864665,79.964291,273.239766,56.231946,0.064253,Stage 2,9
2,600,0.040,8,5.0,19.315817,56.684225,79.894261,272.502008,60.357768,0.057836,Stage 2,1
3,600,0.040,7,5.0,19.340063,56.616044,79.925826,271.352792,46.972423,0.065548,Stage 2,6
4,500,0.040,8,5.0,19.345734,56.502718,80.230336,274.909924,46.921024,0.046824,Stage 2,5
5,800,0.040,8,5.0,19.515506,57.030290,80.010843,274.868721,73.532607,0.062984,Stage 2,4
6,500,0.050,8,5.0,19.560067,57.249586,80.054337,274.996780,47.125830,0.057442,Stage 2,3
7,700,0.035,8,5.0,19.573308,57.002008,80.680630,280.660670,67.707335,0.061783,Stage 2,2
8,600,0.040,8,10.0,19.753354,58.016587,80.797985,281.300992,55.991581,0.061732,Stage 2,10
9,700,0.035,7,7.0,19.838052,57.277113,80.920365,282.505320,55.577962,0.059385,Stage 2,12



BEST CATBOOST STAGE 2 CONFIGURATION


,iterations,learning_rate,depth,l2_leaf_reg,MAE,RMSE,sMAPE,MAPE,training_time_seconds,prediction_time_seconds,stage,config_id
0,600,0.04,9,5.0,19.239601,56.555438,80.110475,274.477167,72.069118,0.139299,Stage 2,7


In [169]:
# ============================================================
# 10.3.7 — Save Best CatBoost Tuning Results
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# Best CatBoost configuration from Stage 2
# ------------------------------------------------------------

best_catboost_row = catboost_stage2_results_df.iloc[0]

CATBOOST_BEST_PARAMS = {
    "iterations": int(best_catboost_row["iterations"]),
    "learning_rate": float(best_catboost_row["learning_rate"]),
    "depth": int(best_catboost_row["depth"]),
    "l2_leaf_reg": float(best_catboost_row["l2_leaf_reg"])
}

CATBOOST_BEST_METRICS = {
    "MAE": float(best_catboost_row["MAE"]),
    "RMSE": float(best_catboost_row["RMSE"]),
    "sMAPE": float(best_catboost_row["sMAPE"]),
    "MAPE": float(best_catboost_row["MAPE"])
}


# ------------------------------------------------------------
# Save directory
# ------------------------------------------------------------

TUNING_DIR = Path(
    r"D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning"
)

TUNING_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Save Stage 2 results
# ------------------------------------------------------------

catboost_stage2_results_path = (
    TUNING_DIR / "catboost_stage2_results.csv"
)

catboost_stage2_results_df.to_csv(
    catboost_stage2_results_path,
    index=False
)


# ------------------------------------------------------------
# Save best CatBoost configuration + metrics
# ------------------------------------------------------------

catboost_best_path = (
    TUNING_DIR / "catboost_best_params.json"
)

catboost_best_record = {
    "model": "CatBoostRegressor",
    "selection_metric": "MAE",
    "secondary_selection_metric": "RMSE",
    "best_params": CATBOOST_BEST_PARAMS,
    "validation_metrics": CATBOOST_BEST_METRICS,
    "categorical_features": categorical_features,
    "test_used_for_tuning": False,
    "validation_used_for_selection": True,
    "random_split": False,
    "shuffle": False,
    "random_seed": 42
}

with open(catboost_best_path, "w") as f:
    json.dump(
        catboost_best_record,
        f,
        indent=4
    )


# ------------------------------------------------------------
# Display confirmation
# ------------------------------------------------------------

print("=" * 70)
print("BEST CATBOOST CONFIGURATION SAVED")
print("=" * 70)

print("\nBest parameters:")
for key, value in CATBOOST_BEST_PARAMS.items():
    print(f"{key}: {value}")

print("\nValidation metrics:")
for key, value in CATBOOST_BEST_METRICS.items():
    print(f"{key}: {value:.6f}")

print("\nCategorical features:")
print(categorical_features)

print("\nSaved files:")
print(f"Stage 2 results: {catboost_stage2_results_path}")
print(f"Best parameters: {catboost_best_path}")

print("\nTest set used for tuning: False")

BEST CATBOOST CONFIGURATION SAVED

Best parameters:
iterations: 600
learning_rate: 0.04
depth: 9
l2_leaf_reg: 5.0

Validation metrics:
MAE: 19.239601
RMSE: 56.555438
sMAPE: 80.110475
MAPE: 274.477167

Categorical features:
['product_id']

Saved files:
Stage 2 results: D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\catboost_stage2_results.csv
Best parameters: D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\catboost_best_params.json

Test set used for tuning: False


In [170]:
# ============================================================
# 10.4.1 — Define Tuned Model Comparison
# ============================================================

TUNED_MODEL_COMPARISON = {
    "HistGradientBoosting": {
        "params": HGB_BEST_PARAMS,
        "MAE": HGB_BEST_METRICS["MAE"],
        "RMSE": HGB_BEST_METRICS["RMSE"],
        "sMAPE": HGB_BEST_METRICS["sMAPE"],
        "MAPE": HGB_BEST_METRICS["MAPE"],
    },
    "CatBoost": {
        "params": CATBOOST_BEST_PARAMS,
        "MAE": CATBOOST_BEST_METRICS["MAE"],
        "RMSE": CATBOOST_BEST_METRICS["RMSE"],
        "sMAPE": CATBOOST_BEST_METRICS["sMAPE"],
        "MAPE": CATBOOST_BEST_METRICS["MAPE"],
    }
}

print("=" * 70)
print("TUNED MODEL COMPARISON SETUP")
print("=" * 70)

for model_name, result in TUNED_MODEL_COMPARISON.items():
    print(f"\n{model_name}")
    print("-" * 40)

    print("Parameters:")
    for key, value in result["params"].items():
        print(f"  {key}: {value}")

    print("\nValidation metrics:")
    print(f"  MAE:   {result['MAE']:.6f}")
    print(f"  RMSE:  {result['RMSE']:.6f}")
    print(f"  sMAPE: {result['sMAPE']:.6f}")
    print(f"  MAPE:  {result['MAPE']:.6f}")

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

TUNED MODEL COMPARISON SETUP

HistGradientBoosting
----------------------------------------
Parameters:
  learning_rate: 0.03
  max_iter: 500
  max_leaf_nodes: 15
  min_samples_leaf: 40
  l2_regularization: 5.0

Validation metrics:
  MAE:   18.737258
  RMSE:  55.416223
  sMAPE: 83.405336
  MAPE:  294.062569

CatBoost
----------------------------------------
Parameters:
  iterations: 600
  learning_rate: 0.04
  depth: 9
  l2_leaf_reg: 5.0

Validation metrics:
  MAE:   19.239601
  RMSE:  56.555438
  sMAPE: 80.110475
  MAPE:  274.477167

TEST SET: NOT USED


In [171]:
# ============================================================
# STEP 10.4.2 — TRAIN TUNED MODELS & VALIDATION PREDICTIONS
# ============================================================

import time
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from catboost import CatBoostRegressor

print("=" * 70)
print("STEP 10.4.2 — TRAIN TUNED MODELS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Train tuned HistGradientBoosting
# ------------------------------------------------------------

print("\nTraining tuned HistGradientBoosting...")

hgb_tuned = HistGradientBoostingRegressor(
    learning_rate=HGB_BEST_PARAMS["learning_rate"],
    max_iter=HGB_BEST_PARAMS["max_iter"],
    max_leaf_nodes=HGB_BEST_PARAMS["max_leaf_nodes"],
    min_samples_leaf=HGB_BEST_PARAMS["min_samples_leaf"],
    l2_regularization=HGB_BEST_PARAMS["l2_regularization"],
    random_state=42
)

hgb_start = time.perf_counter()

hgb_tuned.fit(
    X_train_hgb,
    y_train
)

hgb_training_time = time.perf_counter() - hgb_start

hgb_validation_predictions = hgb_tuned.predict(
    X_validation_hgb
)

print("HistGradientBoosting training complete.")
print(f"Training time: {hgb_training_time:.2f} seconds")
print(f"Validation predictions: {len(hgb_validation_predictions):,}")


# ------------------------------------------------------------
# 2. Train tuned CatBoost
# ------------------------------------------------------------

print("\nTraining tuned CatBoost...")

catboost_tuned = CatBoostRegressor(
    iterations=CATBOOST_BEST_PARAMS["iterations"],
    learning_rate=CATBOOST_BEST_PARAMS["learning_rate"],
    depth=CATBOOST_BEST_PARAMS["depth"],
    l2_leaf_reg=CATBOOST_BEST_PARAMS["l2_leaf_reg"],
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=False,
    thread_count=-1
)

catboost_start = time.perf_counter()

catboost_tuned.fit(
    X_train,
    y_train,
    cat_features=categorical_features
)

catboost_training_time = time.perf_counter() - catboost_start

catboost_validation_predictions = catboost_tuned.predict(
    X_validation
)

print("CatBoost training complete.")
print(f"Training time: {catboost_training_time:.2f} seconds")
print(f"Validation predictions: {len(catboost_validation_predictions):,}")


# ------------------------------------------------------------
# 3. Basic prediction verification
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDATION PREDICTION VERIFICATION")
print("=" * 70)

print(f"\nExpected validation rows: {len(y_validation):,}")

print("\nHistGradientBoosting:")
print(f"  Predictions: {len(hgb_validation_predictions):,}")
print(f"  Min:         {np.min(hgb_validation_predictions):.4f}")
print(f"  Max:         {np.max(hgb_validation_predictions):.4f}")
print(f"  Mean:        {np.mean(hgb_validation_predictions):.4f}")

print("\nCatBoost:")
print(f"  Predictions: {len(catboost_validation_predictions):,}")
print(f"  Min:         {np.min(catboost_validation_predictions):.4f}")
print(f"  Max:         {np.max(catboost_validation_predictions):.4f}")
print(f"  Mean:        {np.mean(catboost_validation_predictions):.4f}")

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

STEP 10.4.2 — TRAIN TUNED MODELS

Training tuned HistGradientBoosting...
HistGradientBoosting training complete.
Training time: 0.70 seconds
Validation predictions: 63,109

Training tuned CatBoost...
CatBoost training complete.
Training time: 75.24 seconds
Validation predictions: 63,109

VALIDATION PREDICTION VERIFICATION

Expected validation rows: 63,109

HistGradientBoosting:
  Predictions: 63,109
  Min:         5.5640
  Max:         365.7059
  Mean:        23.0679

CatBoost:
  Predictions: 63,109
  Min:         -14.4631
  Max:         829.6178
  Mean:        23.0178

TEST SET: NOT USED


In [172]:
# ============================================================
# STEP 10.4.3 — EVALUATE TUNED VALIDATION PREDICTIONS
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("=" * 70)
print("STEP 10.4.3 — TUNED MODEL VALIDATION EVALUATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. HistGradientBoosting metrics
# ------------------------------------------------------------

hgb_tuned_mae = mean_absolute_error(
    y_validation,
    hgb_validation_predictions
)

hgb_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        hgb_validation_predictions
    )
)

hgb_tuned_smape = calculate_smape(
    y_validation,
    hgb_validation_predictions
)

hgb_tuned_mape = calculate_mape(
    y_validation,
    hgb_validation_predictions
)


# ------------------------------------------------------------
# 2. CatBoost metrics
# ------------------------------------------------------------

catboost_tuned_mae = mean_absolute_error(
    y_validation,
    catboost_validation_predictions
)

catboost_tuned_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        catboost_validation_predictions
    )
)

catboost_tuned_smape = calculate_smape(
    y_validation,
    catboost_validation_predictions
)

catboost_tuned_mape = calculate_mape(
    y_validation,
    catboost_validation_predictions
)


# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

print("\nHistGradientBoosting — Tuned")
print("-" * 40)
print(f"MAE:   {hgb_tuned_mae:.6f}")
print(f"RMSE:  {hgb_tuned_rmse:.6f}")
print(f"sMAPE: {hgb_tuned_smape:.6f}")
print(f"MAPE:  {hgb_tuned_mape:.6f}")

print("\nCatBoost — Tuned")
print("-" * 40)
print(f"MAE:   {catboost_tuned_mae:.6f}")
print(f"RMSE:  {catboost_tuned_rmse:.6f}")
print(f"sMAPE: {catboost_tuned_smape:.6f}")
print(f"MAPE:  {catboost_tuned_mape:.6f}")


# ------------------------------------------------------------
# 4. Validation-only comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TUNED MODEL VALIDATION COMPARISON")
print("=" * 70)

print(
    f"\nBest MAE:  "
    f"{'HistGradientBoosting' if hgb_tuned_mae < catboost_tuned_mae else 'CatBoost'}"
)

print(
    f"Best RMSE: "
    f"{'HistGradientBoosting' if hgb_tuned_rmse < catboost_tuned_rmse else 'CatBoost'}"
)

print(
    f"Best sMAPE: "
    f"{'HistGradientBoosting' if hgb_tuned_smape < catboost_tuned_smape else 'CatBoost'}"
)

print(
    f"Best MAPE: "
    f"{'HistGradientBoosting' if hgb_tuned_mape < catboost_tuned_mape else 'CatBoost'}"
)

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

STEP 10.4.3 — TUNED MODEL VALIDATION EVALUATION

HistGradientBoosting — Tuned
----------------------------------------
MAE:   19.115404
RMSE:  55.625835
sMAPE: 84.444981
MAPE:  302.728790

CatBoost — Tuned
----------------------------------------
MAE:   19.239601
RMSE:  56.555438
sMAPE: 80.110475
MAPE:  274.477167

TUNED MODEL VALIDATION COMPARISON

Best MAE:  HistGradientBoosting
Best RMSE: HistGradientBoosting
Best sMAPE: CatBoost
Best MAPE: CatBoost

TEST SET: NOT USED


In [173]:
# ============================================================
# STEP 10.4.4 — FINAL TUNED MODEL VALIDATION COMPARISON
# ============================================================

import pandas as pd

print("=" * 70)
print("STEP 10.4.4 — FINAL VALIDATION MODEL COMPARISON")
print("=" * 70)


# ------------------------------------------------------------
# 1. Best baseline metrics from Step 8
# ------------------------------------------------------------

baseline_name = "Seasonal-Naive-30"

baseline_mae = 36.403285
baseline_rmse = 101.897618
baseline_smape = 88.080540
baseline_mape = 250.741354


# ------------------------------------------------------------
# 2. Original Step 9 ML model results
# ------------------------------------------------------------

original_models = [
    {
        "Model": "HistGradientBoosting",
        "Stage": "Original",
        "MAE": 19.682154,
        "RMSE": 56.531256,
        "sMAPE": 84.290545,
        "MAPE": 304.737432
    },
    {
        "Model": "CatBoost",
        "Stage": "Original",
        "MAE": 19.901269,
        "RMSE": 57.684325,
        "sMAPE": 80.717146,
        "MAPE": 281.336673
    }
]


# ------------------------------------------------------------
# 3. Fresh tuned validation results from Step 10.4.3
# ------------------------------------------------------------

tuned_models = [
    {
        "Model": "HistGradientBoosting",
        "Stage": "Tuned",
        "MAE": hgb_tuned_mae,
        "RMSE": hgb_tuned_rmse,
        "sMAPE": hgb_tuned_smape,
        "MAPE": hgb_tuned_mape
    },
    {
        "Model": "CatBoost",
        "Stage": "Tuned",
        "MAE": catboost_tuned_mae,
        "RMSE": catboost_tuned_rmse,
        "sMAPE": catboost_tuned_smape,
        "MAPE": catboost_tuned_mape
    }
]


# ------------------------------------------------------------
# 4. Create comparison table
# ------------------------------------------------------------

comparison_rows = original_models + tuned_models

validation_comparison = pd.DataFrame(comparison_rows)

validation_comparison = validation_comparison[
    ["Model", "Stage", "MAE", "RMSE", "sMAPE", "MAPE"]
]

print("\nMODEL VALIDATION COMPARISON")
print("-" * 70)

print(
    validation_comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# 5. Calculate tuned improvement over baseline
# ------------------------------------------------------------

tuned_comparison = validation_comparison[
    validation_comparison["Stage"] == "Tuned"
].copy()

tuned_comparison["MAE_improvement_vs_baseline_%"] = (
    (baseline_mae - tuned_comparison["MAE"])
    / baseline_mae
) * 100

tuned_comparison["RMSE_improvement_vs_baseline_%"] = (
    (baseline_rmse - tuned_comparison["RMSE"])
    / baseline_rmse
) * 100

tuned_comparison["sMAPE_improvement_vs_baseline_%"] = (
    (baseline_smape - tuned_comparison["sMAPE"])
    / baseline_smape
) * 100

tuned_comparison["MAPE_improvement_vs_baseline_%"] = (
    (baseline_mape - tuned_comparison["MAPE"])
    / baseline_mape
) * 100


# ------------------------------------------------------------
# 6. Display baseline comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TUNED MODELS VS BEST BASELINE")
print("=" * 70)

print(f"\nBaseline: {baseline_name}")
print(f"  MAE:   {baseline_mae:.6f}")
print(f"  RMSE:  {baseline_rmse:.6f}")
print(f"  sMAPE: {baseline_smape:.6f}")
print(f"  MAPE:  {baseline_mape:.6f}")

print("\nTuned models:")
print(
    tuned_comparison[
        [
            "Model",
            "MAE_improvement_vs_baseline_%",
            "RMSE_improvement_vs_baseline_%",
            "sMAPE_improvement_vs_baseline_%",
            "MAPE_improvement_vs_baseline_%"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}%"
    )
)


# ------------------------------------------------------------
# 7. Identify winners using primary metrics
# ------------------------------------------------------------

best_mae_model = tuned_comparison.loc[
    tuned_comparison["MAE"].idxmin(),
    "Model"
]

best_rmse_model = tuned_comparison.loc[
    tuned_comparison["RMSE"].idxmin(),
    "Model"
]

best_smape_model = tuned_comparison.loc[
    tuned_comparison["sMAPE"].idxmin(),
    "Model"
]

best_mape_model = tuned_comparison.loc[
    tuned_comparison["MAPE"].idxmin(),
    "Model"
]

print("\n" + "=" * 70)
print("FINAL VALIDATION WINNERS")
print("=" * 70)

print(f"\nBest MAE:   {best_mae_model}")
print(f"Best RMSE:  {best_rmse_model}")
print(f"Best sMAPE: {best_smape_model}")
print(f"Best MAPE:  {best_mape_model}")

print("\nPrimary model based on MAE + RMSE:")
print(f"  {best_mae_model}")

print("\nSecondary model:")
print(
    "  CatBoost"
    if best_mae_model == "HistGradientBoosting"
    else "HistGradientBoosting"
)

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

STEP 10.4.4 — FINAL VALIDATION MODEL COMPARISON

MODEL VALIDATION COMPARISON
----------------------------------------------------------------------
               Model    Stage       MAE      RMSE     sMAPE       MAPE
HistGradientBoosting Original 19.682154 56.531256 84.290545 304.737432
            CatBoost Original 19.901269 57.684325 80.717146 281.336673
HistGradientBoosting    Tuned 19.115404 55.625835 84.444981 302.728790
            CatBoost    Tuned 19.239601 56.555438 80.110475 274.477167

TUNED MODELS VS BEST BASELINE

Baseline: Seasonal-Naive-30
  MAE:   36.403285
  RMSE:  101.897618
  sMAPE: 88.080540
  MAPE:  250.741354

Tuned models:
               Model  MAE_improvement_vs_baseline_%  RMSE_improvement_vs_baseline_%  sMAPE_improvement_vs_baseline_%  MAPE_improvement_vs_baseline_%
HistGradientBoosting                         47.49%                          45.41%                            4.13%                         -20.73%
            CatBoost                         4

### Save Final Validation Model Selection

In [174]:
# ============================================================
# STEP 10.5.1 — SAVE FINAL MODEL SELECTION
# ============================================================

import os
import json
import pandas as pd

print("=" * 70)
print("STEP 10.5.1 — SAVE FINAL MODEL SELECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define model-selection directory
# ------------------------------------------------------------

MODEL_DIR = r"D:\All ML Projects\Retail_Demand_Forecasting\data\models"
TUNING_DIR = os.path.join(MODEL_DIR, "hyperparameter_tuning")

os.makedirs(TUNING_DIR, exist_ok=True)


# ------------------------------------------------------------
# 2. Final selected models
# ------------------------------------------------------------

FINAL_PRIMARY_MODEL = "HistGradientBoosting"
FINAL_SECONDARY_MODEL = "CatBoost"


# ------------------------------------------------------------
# 3. Final model configuration
# ------------------------------------------------------------

final_model_selection = {
    "primary_model": FINAL_PRIMARY_MODEL,
    "secondary_model": FINAL_SECONDARY_MODEL,

    "selection_policy": {
        "primary_metrics": ["MAE", "RMSE"],
        "secondary_metric": "sMAPE",
        "diagnostic_metric": "MAPE",
        "validation_used_for_selection": True,
        "test_used_for_tuning": False,
        "random_split": False,
        "shuffle": False,
        "random_state": 42
    },

    "models": {
        "HistGradientBoosting": {
            "parameters": HGB_BEST_PARAMS,
            "validation_metrics": {
                "MAE": float(hgb_tuned_mae),
                "RMSE": float(hgb_tuned_rmse),
                "sMAPE": float(hgb_tuned_smape),
                "MAPE": float(hgb_tuned_mape)
            }
        },

        "CatBoost": {
            "parameters": CATBOOST_BEST_PARAMS,
            "validation_metrics": {
                "MAE": float(catboost_tuned_mae),
                "RMSE": float(catboost_tuned_rmse),
                "sMAPE": float(catboost_tuned_smape),
                "MAPE": float(catboost_tuned_mape)
            }
        }
    },

    "test_set_status": "NOT USED"
}


# ------------------------------------------------------------
# 4. Save final selection
# ------------------------------------------------------------

selection_path = os.path.join(
    TUNING_DIR,
    "final_model_selection.json"
)

with open(selection_path, "w") as f:
    json.dump(
        final_model_selection,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 5. Save validation comparison
# ------------------------------------------------------------

comparison_path = os.path.join(
    TUNING_DIR,
    "final_validation_comparison.csv"
)

validation_comparison.to_csv(
    comparison_path,
    index=False
)


# ------------------------------------------------------------
# 6. Verification
# ------------------------------------------------------------

print("\nFinal model selection:")
print(f"  Primary model:   {FINAL_PRIMARY_MODEL}")
print(f"  Secondary model: {FINAL_SECONDARY_MODEL}")

print("\nSaved files:")
print(f"  {selection_path}")
print(f"  {comparison_path}")

print("\nFile verification:")
print(f"  Selection JSON exists: {os.path.exists(selection_path)}")
print(f"  Comparison CSV exists: {os.path.exists(comparison_path)}")

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

STEP 10.5.1 — SAVE FINAL MODEL SELECTION

Final model selection:
  Primary model:   HistGradientBoosting
  Secondary model: CatBoost

Saved files:
  D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\final_model_selection.json
  D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\final_validation_comparison.csv

File verification:
  Selection JSON exists: True
  Comparison CSV exists: True

TEST SET: NOT USED


### Save Trained Tuned Model Objects

In [175]:
# ============================================================
# STEP 10.5.2 — SAVE TRAINED TUNED MODEL OBJECTS
# ============================================================

import os
import joblib

print("=" * 70)
print("STEP 10.5.2 — SAVE TRAINED TUNED MODEL OBJECTS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Define model save directory
# ------------------------------------------------------------

MODEL_DIR = r"D:\All ML Projects\Retail_Demand_Forecasting\data\models"
TUNING_DIR = os.path.join(MODEL_DIR, "hyperparameter_tuning")

os.makedirs(TUNING_DIR, exist_ok=True)


# ------------------------------------------------------------
# 2. Save tuned HistGradientBoosting model
# ------------------------------------------------------------

hgb_model_path = os.path.join(
    TUNING_DIR,
    "hgb_tuned_model.joblib"
)

joblib.dump(
    hgb_tuned,
    hgb_model_path
)


# ------------------------------------------------------------
# 3. Save tuned CatBoost model
# ------------------------------------------------------------

catboost_model_path = os.path.join(
    TUNING_DIR,
    "catboost_tuned_model.cbm"
)

catboost_tuned.save_model(
    catboost_model_path
)


# ------------------------------------------------------------
# 4. Verify saved files
# ------------------------------------------------------------

print("\nSaved model files:")
print(f"  HistGradientBoosting: {hgb_model_path}")
print(f"  CatBoost:             {catboost_model_path}")

print("\nFile verification:")
print(
    f"  HGB model exists:       "
    f"{os.path.exists(hgb_model_path)}"
)

print(
    f"  CatBoost model exists:  "
    f"{os.path.exists(catboost_model_path)}"
)


# ------------------------------------------------------------
# 5. Confirm models remain trained
# ------------------------------------------------------------

print("\nModel verification:")

print(
    f"  HGB model type: "
    f"{type(hgb_tuned).__name__}"
)

print(
    f"  CatBoost model type: "
    f"{type(catboost_tuned).__name__}"
)

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

STEP 10.5.2 — SAVE TRAINED TUNED MODEL OBJECTS

Saved model files:
  HistGradientBoosting: D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\hgb_tuned_model.joblib
  CatBoost:             D:\All ML Projects\Retail_Demand_Forecasting\data\models\hyperparameter_tuning\catboost_tuned_model.cbm

File verification:
  HGB model exists:       True
  CatBoost model exists:  True

Model verification:
  HGB model type: HistGradientBoostingRegressor
  CatBoost model type: CatBoostRegressor

TEST SET: NOT USED


In [176]:
# ============================================================
# STEP 10.5.3 — RELOAD SAVED MODELS & VERIFY PREDICTIONS
# ============================================================

import os
import joblib
import numpy as np

from catboost import CatBoostRegressor

print("=" * 70)
print("STEP 10.5.3 — RELOAD SAVED MODELS & VALIDATE REPRODUCIBILITY")
print("=" * 70)


# ------------------------------------------------------------
# 1. Define saved model paths
# ------------------------------------------------------------

MODEL_DIR = r"D:\All ML Projects\Retail_Demand_Forecasting\data\models"
TUNING_DIR = os.path.join(MODEL_DIR, "hyperparameter_tuning")

hgb_model_path = os.path.join(
    TUNING_DIR,
    "hgb_tuned_model.joblib"
)

catboost_model_path = os.path.join(
    TUNING_DIR,
    "catboost_tuned_model.cbm"
)


# ------------------------------------------------------------
# 2. Verify model files exist
# ------------------------------------------------------------

print("\nChecking saved model files...")

print(f"HGB file exists:       {os.path.exists(hgb_model_path)}")
print(f"CatBoost file exists:  {os.path.exists(catboost_model_path)}")


# ------------------------------------------------------------
# 3. Reload HistGradientBoosting
# ------------------------------------------------------------

print("\nReloading HistGradientBoosting...")

hgb_reloaded = joblib.load(
    hgb_model_path
)

hgb_reloaded_predictions = hgb_reloaded.predict(
    X_validation_hgb
)

print("HistGradientBoosting reloaded successfully.")


# ------------------------------------------------------------
# 4. Reload CatBoost
# ------------------------------------------------------------

print("\nReloading CatBoost...")

catboost_reloaded = CatBoostRegressor()

catboost_reloaded.load_model(
    catboost_model_path
)

catboost_reloaded_predictions = catboost_reloaded.predict(
    X_validation
)

print("CatBoost reloaded successfully.")


# ------------------------------------------------------------
# 5. Compare original vs reloaded predictions
# ------------------------------------------------------------

hgb_max_difference = np.max(
    np.abs(
        hgb_validation_predictions -
        hgb_reloaded_predictions
    )
)

catboost_max_difference = np.max(
    np.abs(
        catboost_validation_predictions -
        catboost_reloaded_predictions
    )
)


print("\n" + "=" * 70)
print("PREDICTION REPRODUCIBILITY CHECK")
print("=" * 70)

print("\nHistGradientBoosting:")
print(f"  Original predictions:  {len(hgb_validation_predictions):,}")
print(f"  Reloaded predictions:  {len(hgb_reloaded_predictions):,}")
print(f"  Maximum difference:    {hgb_max_difference:.12f}")

print("\nCatBoost:")
print(f"  Original predictions:  {len(catboost_validation_predictions):,}")
print(f"  Reloaded predictions:  {len(catboost_reloaded_predictions):,}")
print(f"  Maximum difference:    {catboost_max_difference:.12f}")


# ------------------------------------------------------------
# 6. Final reproducibility status
# ------------------------------------------------------------

hgb_reproducible = np.allclose(
    hgb_validation_predictions,
    hgb_reloaded_predictions,
    rtol=1e-10,
    atol=1e-10
)

catboost_reproducible = np.allclose(
    catboost_validation_predictions,
    catboost_reloaded_predictions,
    rtol=1e-10,
    atol=1e-10
)

print("\n" + "=" * 70)
print("REPRODUCIBILITY STATUS")
print("=" * 70)

print(f"\nHistGradientBoosting: {hgb_reproducible}")
print(f"CatBoost:             {catboost_reproducible}")

print("\n" + "=" * 70)
print("TEST SET: NOT USED")
print("=" * 70)

STEP 10.5.3 — RELOAD SAVED MODELS & VALIDATE REPRODUCIBILITY

Checking saved model files...
HGB file exists:       True
CatBoost file exists:  True

Reloading HistGradientBoosting...
HistGradientBoosting reloaded successfully.

Reloading CatBoost...
CatBoost reloaded successfully.

PREDICTION REPRODUCIBILITY CHECK

HistGradientBoosting:
  Original predictions:  63,109
  Reloaded predictions:  63,109
  Maximum difference:    0.000000000000

CatBoost:
  Original predictions:  63,109
  Reloaded predictions:  63,109
  Maximum difference:    0.000000000000

REPRODUCIBILITY STATUS

HistGradientBoosting: True
CatBoost:             True

TEST SET: NOT USED


In [177]:
# ============================================================
# STEP 10.6.1 — FINAL TEST-EVALUATION READINESS CHECK
# ============================================================

import os
import numpy as np

print("=" * 70)
print("STEP 10.6.1 — FINAL TEST-EVALUATION READINESS CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Verify required split objects exist
# ------------------------------------------------------------

required_objects = [
    "X_train",
    "y_train",
    "X_validation",
    "y_validation",
    "X_test",
    "y_test",
    "X_train_hgb",
    "X_validation_hgb",
    "hgb_reloaded",
    "catboost_reloaded",
    "FINAL_PRIMARY_MODEL",
    "FINAL_SECONDARY_MODEL"
]

print("\nRequired objects:")

missing_objects = []

for obj_name in required_objects:
    exists = obj_name in globals()

    print(f"  {obj_name}: {exists}")

    if not exists:
        missing_objects.append(obj_name)


# ------------------------------------------------------------
# 2. Verify test split dimensions
# ------------------------------------------------------------

print("\nTest split verification:")

print(f"  X_test shape: {X_test.shape}")
print(f"  y_test shape: {y_test.shape}")

test_shape_valid = (
    len(X_test) == len(y_test)
)

print(f"  Matching row count: {test_shape_valid}")


# ------------------------------------------------------------
# 3. Verify chronological split boundaries
# ------------------------------------------------------------

print("\nChronological split verification:")

print(f"  Train rows:      {len(X_train):,}")
print(f"  Validation rows: {len(X_validation):,}")
print(f"  Test rows:       {len(X_test):,}")

print("\nExpected boundaries:")
print("  Train:      2014-01-01 → 2015-05-25")
print("  Validation: 2015-05-26 → 2015-09-11")
print("  Test:       2015-09-12 → 2015-12-30")


# ------------------------------------------------------------
# 4. Verify final model selection
# ------------------------------------------------------------

print("\nFinal model selection:")

print(f"  Primary model:   {FINAL_PRIMARY_MODEL}")
print(f"  Secondary model: {FINAL_SECONDARY_MODEL}")

selection_valid = (
    FINAL_PRIMARY_MODEL == "HistGradientBoosting"
    and
    FINAL_SECONDARY_MODEL == "CatBoost"
)

print(f"  Selection valid: {selection_valid}")


# ------------------------------------------------------------
# 5. Verify saved model files
# ------------------------------------------------------------

MODEL_DIR = r"D:\All ML Projects\Retail_Demand_Forecasting\data\models"
TUNING_DIR = os.path.join(
    MODEL_DIR,
    "hyperparameter_tuning"
)

hgb_model_path = os.path.join(
    TUNING_DIR,
    "hgb_tuned_model.joblib"
)

catboost_model_path = os.path.join(
    TUNING_DIR,
    "catboost_tuned_model.cbm"
)

print("\nSaved model artifacts:")

print(
    f"  HGB model:       "
    f"{os.path.exists(hgb_model_path)}"
)

print(
    f"  CatBoost model:  "
    f"{os.path.exists(catboost_model_path)}"
)


# ------------------------------------------------------------
# 6. Verify test set has not been used for tuning
# ------------------------------------------------------------

test_used_for_tuning = False

print("\nTest-set usage policy:")
print(f"  Test used for tuning: {test_used_for_tuning}")
print("  Test used for model selection: False")


# ------------------------------------------------------------
# 7. Final readiness status
# ------------------------------------------------------------

ready_for_test_evaluation = (
    len(missing_objects) == 0
    and test_shape_valid
    and selection_valid
    and os.path.exists(hgb_model_path)
    and os.path.exists(catboost_model_path)
    and test_used_for_tuning is False
)

print("\n" + "=" * 70)
print("FINAL READINESS STATUS")
print("=" * 70)

if ready_for_test_evaluation:
    print("\nREADY FOR FINAL TEST EVALUATION: TRUE")
else:
    print("\nREADY FOR FINAL TEST EVALUATION: FALSE")

    if missing_objects:
        print("\nMissing objects:")
        for obj in missing_objects:
            print(f"  - {obj}")

print("\n" + "=" * 70)
print("IMPORTANT: THIS CELL DOES NOT GENERATE TEST PREDICTIONS")
print("=" * 70)

STEP 10.6.1 — FINAL TEST-EVALUATION READINESS CHECK

Required objects:
  X_train: True
  y_train: True
  X_validation: True
  y_validation: True
  X_test: True
  y_test: True
  X_train_hgb: True
  X_validation_hgb: True
  hgb_reloaded: True
  catboost_reloaded: True
  FINAL_PRIMARY_MODEL: True
  FINAL_SECONDARY_MODEL: True

Test split verification:
  X_test shape: (81521, 25)
  y_test shape: (81521,)
  Matching row count: True

Chronological split verification:
  Train rows:      103,272
  Validation rows: 63,109
  Test rows:       81,521

Expected boundaries:
  Train:      2014-01-01 → 2015-05-25
  Validation: 2015-05-26 → 2015-09-11
  Test:       2015-09-12 → 2015-12-30

Final model selection:
  Primary model:   HistGradientBoosting
  Secondary model: CatBoost
  Selection valid: True

Saved model artifacts:
  HGB model:       True
  CatBoost model:  True

Test-set usage policy:
  Test used for tuning: False
  Test used for model selection: False

FINAL READINESS STATUS

READY FOR FIN